In [3]:
# ============================================================
# CELL 1 — Install packages + environment setup
# ============================================================

!pip install ultralytics --upgrade -q
!pip install realesrgan basicsr -q
!pip install pycocotools -q

# ============================================================
# Patch basicsr torchvision compatibility
# ============================================================

import pathlib
import site

for sp in site.getsitepackages():

    deg = pathlib.Path(sp) / "basicsr/data/degradations.py"

    if deg.exists():

        txt = deg.read_text()

        old = (
            "from torchvision.transforms"
            ".functional_tensor import rgb_to_grayscale"
        )

        new = (
            "from torchvision.transforms"
            ".functional import rgb_to_grayscale"
        )

        if old in txt:

            deg.write_text(
                txt.replace(old, new)
            )

            print("✅ basicsr patched")

        else:

            print("✅ basicsr already patched")

        break

# ============================================================
# Imports
# ============================================================

import os
import gc
import cv2
import torch
import shutil
import numpy as np

# ============================================================
# CUDA setup
# ============================================================

torch.backends.cudnn.benchmark = True

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()

# ============================================================
# Device
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

# ============================================================
# System info
# ============================================================

print(f"\n✅ PyTorch : {torch.__version__}")

if torch.cuda.is_available():

    print(
        f"✅ GPU     : "
        f"{torch.cuda.get_device_name(0)}"
    )

    print(
        f"✅ VRAM    : "
        f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB"
    )

# ============================================================
# Disk info
# ============================================================

_, used, free = shutil.disk_usage("/kaggle/working")

print(
    f"✅ Disk    : "
    f"{used/1e9:.1f}GB used | "
    f"{free/1e9:.1f}GB free"
)

# ============================================================
# Test ESRGAN imports
# ============================================================

from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet
from basicsr.utils.download_util import load_file_from_url

print("\n✅ ESRGAN imports working")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 19.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 91.3 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires nu

In [4]:
# ============================================================
# CELL 2 — All paths, config, imports
# MODIFIED: correct AI-TOD path + verified VisDrone path
# ============================================================

import os, cv2, sys, json, shutil, random, yaml, math
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

# ── Fix PyTorch 2.6 ──────────────────────────────────────────
if not hasattr(torch, "_load_patched"):
    _orig_load = torch.load
    def patched_load(f, *args, **kwargs):
        kwargs["weights_only"] = False
        return _orig_load(f, *args, **kwargs)
    torch.load = patched_load
    torch._load_patched = True

# ── Reproducibility ───────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── Device ────────────────────────────────────────────────────
DEVICE     = torch.device("cuda" if torch.cuda.is_available()
                          else "cpu")
device_obj = DEVICE

# ── VisDrone tiny source ──────────────────────────────────────
VD_ROOT      = Path("/kaggle/input/datasets/arnavdsp/datasets-tiny/filtered_tiny-20260522T184816Z-3-001/filtered_tiny")
VD_TRAIN_IMG = VD_ROOT / "images"
VD_TRAIN_LBL = VD_ROOT / "labels"
VD_VAL_IMG   = VD_ROOT / "val_images"
VD_VAL_LBL   = VD_ROOT / "val_labels"

# ── AI-TOD source ─────────────────────────────────────────────
# FIXED: user confirmed path ends at /train
AITOD_TRAIN  = Path("/kaggle/input/datasets/arnavdsp/datasets-tiny/aitod_sampled-20260603T115655Z-3-001/aitod_sampled/train")

# ── Working directories ───────────────────────────────────────
WORK_DIR     = Path("/kaggle/working")
WEIGHTS_DIR  = WORK_DIR / "esrgan_weights"
VD_TILES     = WORK_DIR / "vd_tiles"
AT_TILES     = WORK_DIR / "aitod_tiles"
RUNS_DIR     = WORK_DIR / "runs"

for d in [WEIGHTS_DIR, VD_TILES, AT_TILES, RUNS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Class definitions ─────────────────────────────────────────
VD_CLASSES = ["pedestrian","people","bicycle",
              "car","tricycle","motor"]
VD_NC      = len(VD_CLASSES)

AT_CLASSES = ["airplane","bridge","storage-tank","ship",
              "swimming-pool","vehicle","person","wind-mill"]
AT_NC      = len(AT_CLASSES)

# ── VisDrone raw class ID → YOLO index ───────────────────────
# Used during SAHI slicing (Cell 5)
VD_ID_TO_YOLO = {1:0, 2:1, 3:2, 4:3, 7:4, 10:5}
VD_KEEP_IDS   = set(VD_ID_TO_YOLO.keys())

# ── AI-TOD: detect annotation format during Cell 4 ───────────
# (YOLO txt or COCO json — set after Cell 4 runs)
AT_FORMAT = None   # filled by Cell 4

# ── Hyperparameters ───────────────────────────────────────────
IMG_SIZE    = 640
BATCH_SIZE  = 1
GRAD_ACCUM  = 4
NUM_WORKERS = 2
EPOCHS_2X   = 10
EPOCHS_4X   = 3
LR_ESRGAN   = 1e-4
LR_DETECT   = 1e-4
LAMBDA_SR   = 0.1
NWD_C       = 12.8
MAX_BOXES   = 50
NUM_QUERIES = 100
TILE_H = TILE_W = 640
OVERLAP  = 0.2
MIN_AREA = 0.5

print("✅ Cell 2 config ready")
print(f"   VisDrone  : {VD_ROOT}")
print(f"   AI-TOD    : {AITOD_TRAIN}")
print(f"   Device    : {DEVICE}")

# Verify both source paths exist
for name, path in [("VisDrone", VD_ROOT),
                   ("AI-TOD",   AITOD_TRAIN)]:
    exists = path.exists()
    n = len(list(path.rglob("*"))) if exists else 0
    print(f"   {'✅' if exists else '❌'} {name}: "
          f"{'exists' if exists else 'NOT FOUND'} "
          f"({n} items)")

✅ Cell 2 config ready
   VisDrone  : /kaggle/input/datasets/arnavdsp/datasets-tiny/filtered_tiny-20260522T184816Z-3-001/filtered_tiny
   AI-TOD    : /kaggle/input/datasets/arnavdsp/datasets-tiny/aitod_sampled-20260603T115655Z-3-001/aitod_sampled/train
   Device    : cuda
   ✅ VisDrone: exists (1440 items)
   ✅ AI-TOD: exists (3994 items)


In [5]:
# ============================================================
# CELL 3 — Verify VisDrone tiny dataset (FIXED)
# ============================================================

import cv2
from pathlib import Path
from collections import defaultdict

VD_ROOT = Path(
    "/kaggle/input/datasets/arnavdsp/datasets-tiny/filtered_tiny-20260522T184816Z-3-001/filtered_tiny"
)

print("=" * 55)
print("  VisDrone Tiny Dataset Check")
print("=" * 55)

checks = {
    "train images": VD_ROOT / "images",
    "train labels": VD_ROOT / "labels",
    "val images": VD_ROOT / "val_images",
    "val labels": VD_ROOT / "val_labels",
}

# ------------------------------------------------------------
# Folder checks
# ------------------------------------------------------------

for name, d in checks.items():

    if d.exists():

        n = (
            len(list(d.glob("*.jpg"))) +
            len(list(d.glob("*.png"))) +
            len(list(d.glob("*.txt")))
        )

    else:
        n = 0

    status = "✅" if n > 0 else "❌"

    print(f"  {status}  {name:<15}: {n} files  [{d.name}]")

# ------------------------------------------------------------
# Sample label check
# ------------------------------------------------------------

sample_lbl = next((VD_ROOT / "labels").glob("*.txt"), None)

if sample_lbl:

    lines = sample_lbl.read_text().strip().split("\n")

    print(f"\n  Sample label ({sample_lbl.name}):")

    for l in lines[:3]:
        print(f"    '{l}'")

    cls_ids = set()

    for l in lines:

        # VisDrone format:
        # x,y,w,h,score,class,truncation,occlusion

        p = l.strip().split(",")

        if len(p) >= 6:

            cls_id = int(p[5])
            cls_ids.add(cls_id)

    print(f"\n  Class IDs in this file: {sorted(cls_ids)}")

# ------------------------------------------------------------
# Sample image size
# ------------------------------------------------------------

sample_img = next((VD_ROOT / "images").glob("*.jpg"), None)

if sample_img:

    img = cv2.imread(str(sample_img))

    print(
        f"\n  Sample image size: "
        f"{img.shape[1]} × {img.shape[0]}"
    )

# ------------------------------------------------------------
# Class distribution
# ------------------------------------------------------------

print(f"\n  Class distribution (train):")

VISDRONE_CLASSES = {
    1: "pedestrian",
    2: "people",
    3: "bicycle",
    4: "car",
    5: "van",
    6: "truck",
    7: "tricycle",
    8: "awning-tricycle",
    9: "bus",
    10: "motor"
}

counts = defaultdict(int)

for lbl in (VD_ROOT / "labels").glob("*.txt"):

    lines = lbl.read_text().strip().split("\n")

    for line in lines:

        p = line.strip().split(",")

        if len(p) >= 6:

            cls_id = int(p[5])

            counts[cls_id] += 1

# Print distribution
for cls_id, cls_name in VISDRONE_CLASSES.items():

    print(
        f"    {cls_id:>2} "
        f"{cls_name:<18}: "
        f"{counts[cls_id]:>6,}"
    )

print("=" * 55)

  VisDrone Tiny Dataset Check
  ✅  train images   : 658 files  [images]
  ✅  train labels   : 658 files  [labels]
  ✅  val images     : 60 files  [val_images]
  ✅  val labels     : 60 files  [val_labels]

  Sample label (0000348_02157_d_0000417.txt):
    '234,358,17,37,1,1,0,0'
    '248,377,11,22,1,1,0,0'
    '650,149,11,20,1,1,0,0'

  Class IDs in this file: [1, 2]

  Sample image size: 1400 × 1050

  Class distribution (train):
     1 pedestrian        : 13,027
     2 people            :  2,519
     3 bicycle           :    841
     4 car               :  4,236
     5 van               :      0
     6 truck             :      0
     7 tricycle          :    161
     8 awning-tricycle   :      0
     9 bus               :      0
    10 motor             :  1,435


In [6]:
# ============================================================
# CELL 4 — Inspect AI-TOD dataset structure
# MODIFIED: uses correct AITOD_TRAIN path
#           detects annotation format automatically
# ============================================================

from pathlib import Path
import json

AITOD_TRAIN = Path("/kaggle/input/datasets/arnavdsp/datasets-tiny/aitod_sampled-20260603T115655Z-3-001/aitod_sampled/train")

print("="*65)
print(f"AI-TOD TRAIN: {AITOD_TRAIN}")
print("="*65)

if not AITOD_TRAIN.exists():
    print("❌ Path does not exist!")
    print("   Check dataset is attached to this notebook")
else:
    # Show all items (depth limited)
    all_items = sorted(AITOD_TRAIN.rglob("*"))
    print(f"Total items: {len(all_items)}\n")
    for p in all_items[:40]:
        typ = "DIR " if p.is_dir() else "FILE"
        print(f"  {typ} {p.relative_to(AITOD_TRAIN)}")
    if len(all_items) > 40:
        print(f"  ... and {len(all_items)-40} more")

    # ── Detect format ─────────────────────────────────────────
    print("\n" + "="*65)
    print("FORMAT DETECTION")
    print("="*65)

    json_files = list(AITOD_TRAIN.rglob("*.json"))
    txt_files  = list(AITOD_TRAIN.rglob("*.txt"))
    img_files  = (list(AITOD_TRAIN.rglob("*.jpg")) +
                  list(AITOD_TRAIN.rglob("*.png")))

    print(f"  JSON files : {len(json_files)}")
    print(f"  TXT files  : {len(txt_files)}")
    print(f"  Images     : {len(img_files)}")

    # COCO JSON format
    if json_files:
        print("\n  → COCO JSON format detected")
        with open(json_files[0]) as f:
            coco = json.load(f)
        print(f"  Images      : {len(coco.get('images',[]))}")
        print(f"  Annotations : {len(coco.get('annotations',[]))}")
        cats = coco.get("categories",[])
        print(f"  Categories  : {[(c['id'],c['name']) for c in cats]}")
        print(f"\n  JSON path   : {json_files[0]}")
        AT_FORMAT = "coco"

    # YOLO TXT format
    elif txt_files:
        sample = txt_files[0]
        sample_line = sample.read_text().strip().split("\n")[0]
        print(f"\n  → YOLO TXT format detected")
        print(f"  Sample line : '{sample_line}'")
        parts = sample_line.strip().split()
        if len(parts) == 5:
            print("  ✅ Confirmed: class cx cy w h format")
        AT_FORMAT = "yolo"
    else:
        print("\n  ❌ No annotation files found")
        AT_FORMAT = None

    # Image folder
    if img_files:
        import cv2
        sample_img = cv2.imread(str(img_files[0]))
        if sample_img is not None:
            print(f"\n  Sample image: {img_files[0].name}")
            print(f"  Image size  : {sample_img.shape[1]}"
                  f"×{sample_img.shape[0]}")

print(f"\n✅ AT_FORMAT = '{AT_FORMAT}'")
print("   (used by Cell 5 SAHI slicer)")

AI-TOD TRAIN: /kaggle/input/datasets/arnavdsp/datasets-tiny/aitod_sampled-20260603T115655Z-3-001/aitod_sampled/train
Total items: 3994

  DIR  images
  FILE images/0000042_02231_d_0000075__160_0_png.rf.50468a58af59a28c100bef261c5fda8c.jpg
  FILE images/0000043_00500_d_0000077__0_0_png.rf.ea494cd873f88155ade767fcb68970aa.jpg
  FILE images/0000101_01577_d_0000018__160_0_png.rf.7d90521e557453a3e2485fcfd48354bd.jpg
  FILE images/0000130_01308_d_0000139__1200_0_png.rf.514f5a65108322e65ab9bcd7b0051986.jpg
  FILE images/0000142_04458_d_0000045__1120_0_png.rf.c70191bce6d7b3e9df1a75bf749337a4.jpg
  FILE images/0000143_00681_d_0000051__1120_0_png.rf.148495fb41e4df63bebeda0668bc9307.jpg
  FILE images/0000154_00001_d_0000001__160_0_png.rf.551fe65441efffe8549758ecd160563c.jpg
  FILE images/0000170_00001_d_0000001__0_0_png.rf.e8ed3fea099dac5f7db6c1614f2772e1.jpg
  FILE images/0000170_00801_d_0000001__160_0_png.rf.362d4343a1e0c7664a56e2ca5dca507a.jpg
  FILE images/0000170_01201_d_0000001__160_0_png.r

In [7]:
# ============================================================
# CELL 5 — SAHI Slicing for VisDrone + AI-TOD
# MODIFIED:
#   ✅ VD uses raw VisDrone CSV format (x,y,w,h,score,cls...)
#   ✅ AT uses detected format (COCO JSON or YOLO TXT)
#   ✅ Both output clean YOLO-format tiles
#   ✅ Only tiles WITH objects are saved (no empty tiles)
# ============================================================

import cv2, shutil, json
from pathlib import Path
from collections import defaultdict
from tqdm.notebook import tqdm

VD_ROOT     = Path("/kaggle/input/datasets/arnavdsp/datasets-tiny/filtered_tiny-20260522T184816Z-3-001/filtered_tiny")
AITOD_TRAIN = Path("/kaggle/input/datasets/arnavdsp/datasets-tiny/aitod_sampled-20260603T115655Z-3-001/aitod_sampled/train")
VD_TILES    = Path("/kaggle/working/vd_tiles")
AT_TILES    = Path("/kaggle/working/aitod_tiles")

TILE_H   = TILE_W = 640
OVERLAP  = 0.2
MIN_AREA = 0.5

# VisDrone: keep only these raw class IDs
VD_ID_TO_YOLO = {1:0, 2:1, 3:2, 4:3, 7:4, 10:5}
VD_KEEP_IDS   = set(VD_ID_TO_YOLO.keys())

# AI-TOD: COCO category_id (1-indexed) → YOLO index
AT_CAT_TO_YOLO = {1:0,2:1,3:2,4:3,5:4,6:5,7:6,8:7}


# ============================================================
# Label parsers → return list of [yolo_cls, x1, y1, x2, y2]
#                 in PIXEL coordinates
# ============================================================

def parse_visdrone_label(lbl_path, W, H):
    """Parse VisDrone CSV annotation → pixel xyxy boxes."""
    boxes = []
    if not lbl_path.exists():
        return boxes
    for line in lbl_path.read_text().strip().split("\n"):
        p = line.strip().split(",")
        if len(p) < 6:
            continue
        score  = int(p[4])
        cls_id = int(p[5])
        if score == 0:           # ignored region
            continue
        if cls_id not in VD_KEEP_IDS:
            continue
        x, y, w, h = float(p[0]),float(p[1]),float(p[2]),float(p[3])
        if w <= 1 or h <= 1:
            continue
        yolo_cls = VD_ID_TO_YOLO[cls_id]
        boxes.append([yolo_cls, x, y, x+w, y+h])
    return boxes


def parse_yolo_label(lbl_path, W, H):
    """Parse YOLO txt annotation → pixel xyxy boxes."""
    boxes = []
    if not lbl_path.exists():
        return boxes
    for line in lbl_path.read_text().strip().split("\n"):
        p = line.strip().split()
        if len(p) != 5:
            continue
        cls = int(p[0])
        cx  = float(p[1]) * W;  cy = float(p[2]) * H
        bw  = float(p[3]) * W;  bh = float(p[4]) * H
        boxes.append([cls, cx-bw/2, cy-bh/2, cx+bw/2, cy+bh/2])
    return boxes


def build_coco_lookup(json_path):
    """Build image_id → list of [yolo_cls,x1,y1,x2,y2] dict."""
    with open(json_path) as f:
        coco = json.load(f)
    # cat_id → yolo_cls
    cats = {c["id"]: AT_CAT_TO_YOLO.get(c["id"], c["id"]-1)
            for c in coco.get("categories",[])}
    lookup = defaultdict(list)
    for ann in coco.get("annotations", []):
        img_id = ann["image_id"]
        cat_id = ann["category_id"]
        x, y, bw, bh = ann["bbox"]
        if bw <= 1 or bh <= 1:
            continue
        yolo_cls = cats.get(cat_id, 0)
        lookup[img_id].append([yolo_cls, x, y, x+bw, y+bh])
    # image_id → filename
    id_to_file = {img["id"]: img["file_name"]
                  for img in coco.get("images",[])}
    return lookup, id_to_file


# ============================================================
# Tile slicer
# ============================================================

def slice_image_to_tiles(img, boxes_xyxy,
                          out_img_dir, out_lbl_dir,
                          stem, tile_h, tile_w,
                          overlap, min_area):
    """Slice one image into tiles. Save only tiles with objects."""
    H, W = img.shape[:2]
    step_h = int(tile_h * (1 - overlap))
    step_w = int(tile_w * (1 - overlap))
    saved  = 0

    for y in range(0, max(1, H-tile_h+step_h), step_h):
        for x in range(0, max(1, W-tile_w+step_w), step_w):
            y2 = min(y+tile_h, H); x2 = min(x+tile_w, W)
            y1 = max(0, y2-tile_h); x1 = max(0, x2-tile_w)
            tile = img[y1:y2, x1:x2]
            th, tw = tile.shape[:2]

            tile_boxes = []
            for (cls, bx1, by1, bx2, by2) in boxes_xyxy:
                # Intersect
                ix1 = max(bx1, x1); iy1 = max(by1, y1)
                ix2 = min(bx2, x2); iy2 = min(by2, y2)
                if ix2 <= ix1 or iy2 <= iy1:
                    continue
                orig  = max(1, (bx2-bx1)*(by2-by1))
                inter = (ix2-ix1)*(iy2-iy1)
                if inter/orig < min_area:
                    continue
                # To normalised YOLO
                ncx = ((ix1+ix2)/2 - x1) / tw
                ncy = ((iy1+iy2)/2 - y1) / th
                nw  = (ix2-ix1) / tw
                nh  = (iy2-iy1) / th
                ncx = max(0.001, min(0.999, ncx))
                ncy = max(0.001, min(0.999, ncy))
                nw  = max(0.001, min(0.999, nw))
                nh  = max(0.001, min(0.999, nh))
                tile_boxes.append(
                    f"{int(cls)} {ncx:.6f} {ncy:.6f}"
                    f" {nw:.6f} {nh:.6f}")

            if not tile_boxes:
                continue    # skip empty tiles

            tstem = f"{stem}_{y1}_{x1}"
            cv2.imwrite(str(out_img_dir/f"{tstem}.jpg"), tile)
            (out_lbl_dir/f"{tstem}.txt").write_text(
                "\n".join(tile_boxes))
            saved += 1

    return saved


# ============================================================
# Run VisDrone slicing (train + val)
# ============================================================

print("="*60)
print("SLICING VISDRONE TINY")
print("="*60)

for split, img_src, lbl_src in [
    ("train", VD_ROOT/"images",     VD_ROOT/"labels"),
    ("val",   VD_ROOT/"val_images", VD_ROOT/"val_labels"),
]:
    dst_img = VD_TILES / split / "images"
    dst_lbl = VD_TILES / split / "labels"
    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)

    imgs   = sorted(img_src.glob("*.jpg"))
    total_tiles = obj_tiles = 0

    print(f"\n[VisDrone {split}] {len(imgs)} images")

    for img_path in tqdm(imgs, desc=f"VD {split}"):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        H, W = img.shape[:2]
        lbl_path = lbl_src / (img_path.stem + ".txt")
        boxes = parse_visdrone_label(lbl_path, W, H)
        if not boxes:
            continue
        n = slice_image_to_tiles(
            img, boxes, dst_img, dst_lbl,
            img_path.stem, TILE_H, TILE_W,
            OVERLAP, MIN_AREA)
        obj_tiles += n

    n_img = len(list(dst_img.glob("*.jpg")))
    print(f"  ✅ {obj_tiles} object tiles saved")


# ============================================================
# Run AI-TOD slicing (train only — no val split in source)
# ============================================================

print("\n" + "="*60)
print("SLICING AI-TOD")
print("="*60)

# Detect format
json_files = list(AITOD_TRAIN.rglob("*.json"))
txt_files  = list(AITOD_TRAIN.rglob("*.txt"))

# Output: train split (use 90/10 split for val)
AT_TRAIN_IMG = AT_TILES / "train" / "images"
AT_TRAIN_LBL = AT_TILES / "train" / "labels"
AT_VAL_IMG   = AT_TILES / "val"   / "images"
AT_VAL_LBL   = AT_TILES / "val"   / "labels"
for d in [AT_TRAIN_IMG, AT_TRAIN_LBL, AT_VAL_IMG, AT_VAL_LBL]:
    d.mkdir(parents=True, exist_ok=True)

import random as rng
rng.seed(42)
all_tiles_stemmed = []    # collect stems for train/val split

if json_files:
    # COCO JSON format
    print(f"  Format: COCO JSON ({json_files[0].name})")
    coco_lookup, id_to_file = build_coco_lookup(json_files[0])

    # Find image directory
    img_dirs = [d for d in AITOD_TRAIN.rglob("*")
                if d.is_dir() and any(d.glob("*.jpg"))]
    img_dir  = img_dirs[0] if img_dirs else AITOD_TRAIN

    print(f"  Image dir  : {img_dir}")
    print(f"  Images with anns: {len(coco_lookup)}")

    # Use a temp dir to collect all tiles before split
    TMP_IMG = AT_TILES / "_tmp" / "images"
    TMP_LBL = AT_TILES / "_tmp" / "labels"
    TMP_IMG.mkdir(parents=True, exist_ok=True)
    TMP_LBL.mkdir(parents=True, exist_ok=True)

    for img_id, boxes in tqdm(coco_lookup.items(),
                               desc="COCO tiles"):
        fname    = id_to_file.get(img_id, "")
        img_path = img_dir / Path(fname).name
        if not img_path.exists():
            candidates = list(img_dir.rglob(Path(fname).name))
            if not candidates:
                continue
            img_path = candidates[0]
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        n = slice_image_to_tiles(
            img, boxes, TMP_IMG, TMP_LBL,
            img_path.stem, TILE_H, TILE_W,
            OVERLAP, MIN_AREA)
        all_tiles_stemmed.extend(
            [p.stem for p in TMP_IMG.glob(f"{img_path.stem}_*.jpg")])

elif txt_files:
    # YOLO TXT format
    img_dirs = [d for d in AITOD_TRAIN.rglob("*")
                if d.is_dir() and any(d.glob("*.jpg"))]
    lbl_dirs = [d for d in AITOD_TRAIN.rglob("*")
                if d.is_dir() and any(d.glob("*.txt"))]
    img_dir  = img_dirs[0] if img_dirs else AITOD_TRAIN
    lbl_dir  = lbl_dirs[0] if lbl_dirs else AITOD_TRAIN

    print(f"  Format: YOLO TXT")
    print(f"  Image dir: {img_dir}")

    TMP_IMG = AT_TILES / "_tmp" / "images"
    TMP_LBL = AT_TILES / "_tmp" / "labels"
    TMP_IMG.mkdir(parents=True, exist_ok=True)
    TMP_LBL.mkdir(parents=True, exist_ok=True)

    for img_path in tqdm(sorted(img_dir.glob("*.jpg")),
                          desc="YOLO tiles"):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        H, W = img.shape[:2]
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        boxes = parse_yolo_label(lbl_path, W, H)
        if not boxes:
            continue
        slice_image_to_tiles(
            img, boxes, TMP_IMG, TMP_LBL,
            img_path.stem, TILE_H, TILE_W,
            OVERLAP, MIN_AREA)
else:
    print("  ❌ No annotations found in AI-TOD path")
    TMP_IMG = None

# ── 90/10 train/val split of all AI-TOD tiles ────────────────
if TMP_IMG and TMP_IMG.exists():
    all_tile_imgs = sorted(TMP_IMG.glob("*.jpg"))
    rng.shuffle(all_tile_imgs)
    n_val = max(1, int(len(all_tile_imgs) * 0.1))
    val_stems = {p.stem for p in all_tile_imgs[:n_val]}

    for tile_img in all_tile_imgs:
        tile_lbl = TMP_LBL / (tile_img.stem + ".txt")
        split    = "val" if tile_img.stem in val_stems else "train"
        dst_i    = AT_TILES / split / "images" / tile_img.name
        dst_l    = AT_TILES / split / "labels" / (tile_img.stem+".txt")
        if not dst_i.exists():
            shutil.copy(tile_img, dst_i)
        if tile_lbl.exists() and not dst_l.exists():
            shutil.copy(tile_lbl, dst_l)

    # Cleanup tmp
    shutil.rmtree(AT_TILES / "_tmp", ignore_errors=True)

# ── Summary ───────────────────────────────────────────────────
print("\n" + "="*60)
print("SAHI SLICING COMPLETE")
print("="*60)
for name, base in [("VisDrone", VD_TILES), ("AI-TOD", AT_TILES)]:
    for split in ["train","val"]:
        d = base / split / "images"
        n = len(list(d.glob("*.jpg"))) if d.exists() else 0
        print(f"  {name:<10} [{split}]: {n:>6} tiles")

import shutil as sh
_, used, free = sh.disk_usage("/kaggle/working")
print(f"\n  Disk: {used/1e9:.1f}GB used  {free/1e9:.1f}GB free")

SLICING VISDRONE TINY

[VisDrone train] 658 images


VD train:   0%|          | 0/658 [00:00<?, ?it/s]

  ✅ 3459 object tiles saved

[VisDrone val] 60 images


VD val:   0%|          | 0/60 [00:00<?, ?it/s]

  ✅ 259 object tiles saved

SLICING AI-TOD
  Format: YOLO TXT
  Image dir: /kaggle/input/datasets/arnavdsp/datasets-tiny/aitod_sampled-20260603T115655Z-3-001/aitod_sampled/train/images


YOLO tiles:   0%|          | 0/1996 [00:00<?, ?it/s]


SAHI SLICING COMPLETE
  VisDrone   [train]:   3459 tiles
  VisDrone   [val]:    259 tiles
  AI-TOD     [train]:   5830 tiles
  AI-TOD     [val]:    647 tiles

  Disk: 1.2GB used  19.7GB free


In [8]:
# ============================================================
# CELL 6 — Download ESRGAN weights + define NWD loss classes
# ============================================================

import urllib.request, torch, torch.nn as nn
from pathlib import Path
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet

WEIGHTS_DIR = Path("/kaggle/working/esrgan_weights")
WEIGHTS_DIR.mkdir(exist_ok=True)

ESRGAN_WEIGHTS = {
    "2x": {
        "url"  : "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth",
        "path" : WEIGHTS_DIR / "RealESRGAN_x2plus.pth",
        "scale": 2,
    },
    "4x": {
        "url"  : "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
        "path" : WEIGHTS_DIR / "RealESRGAN_x4plus.pth",
        "scale": 4,
    },
}

for key, cfg in ESRGAN_WEIGHTS.items():
    if not cfg["path"].exists():
        print(f"⬇️  Downloading ESRGAN {key}...")
        urllib.request.urlretrieve(cfg["url"], cfg["path"])
        mb = cfg["path"].stat().st_size / 1e6
        print(f"   ✅ {mb:.0f} MB")
    else:
        mb = cfg["path"].stat().st_size / 1e6
        print(f"✅ ESRGAN {key} ready ({mb:.0f} MB)")


# ── NWD Loss ─────────────────────────────────────────────────
class NWDLoss(nn.Module):
    """
    Normalized Wasserstein Distance loss.
    Designed specifically for tiny object detection.

    Models each box as a 2D Gaussian:
      μ  = (cx, cy)    — centre
      σ  = (w/2, h/2)  — spread

    W2² = ||μ1-μ2||² + ||σ1-σ2||²
    NWD = exp(-sqrt(W2²) / C)
    Loss= 1 - NWD

    Advantage over IoU: smooth gradient even when
    boxes do not overlap (common for tiny objects).
    """
    def __init__(self, C: float = 12.8):
        super().__init__()
        self.C = C

    def forward(self, pred, target,
                reduction="mean"):
        mu1 = pred[:,:2];   sigma1 = pred[:,2:]/2
        mu2 = target[:,:2]; sigma2 = target[:,2:]/2
        center_d = ((mu1-mu2)**2).sum(-1)
        sigma_d  = ((sigma1-sigma2)**2).sum(-1)
        w2       = torch.sqrt(center_d+sigma_d+1e-7)
        nwd      = torch.exp(-w2/self.C)
        loss     = 1.0 - nwd
        if reduction=="mean":  return loss.mean()
        if reduction=="sum":   return loss.sum()
        return loss


class SRQualityLoss(nn.Module):
    """L1 pixel loss + VGG19 perceptual loss for SR quality."""
    def __init__(self, device):
        super().__init__()
        import torchvision.models as tv
        vgg = tv.vgg19(weights=tv.VGG19_Weights.IMAGENET1K_V1)
        self.feat = nn.Sequential(
            *list(vgg.features)[:18]).eval().to(device)
        for p in self.feat.parameters():
            p.requires_grad = False
        self.l1   = nn.L1Loss()
        self.mean = torch.tensor(
            [0.485,0.456,0.406]).view(1,3,1,1).to(device)
        self.std  = torch.tensor(
            [0.229,0.224,0.225]).view(1,3,1,1).to(device)

    def forward(self, sr, hr):
        L_pix  = self.l1(sr, hr)
        sr_n   = (sr-self.mean)/self.std
        hr_n   = (hr-self.mean)/self.std
        L_perc = self.l1(self.feat(sr_n), self.feat(hr_n))
        return L_pix + 0.1*L_perc


def build_esrgan(scale, device):
    """Build trainable ESRGAN model."""
    model = RRDBNet(num_in_ch=3,num_out_ch=3,
                    num_feat=64,num_block=23,
                    num_grow_ch=32,scale=scale)
    upsampler = RealESRGANer(
        scale      = scale,
        model_path = str(ESRGAN_WEIGHTS[f"{scale}x"]["path"]),
        model      = model,
        tile       = 0,
        tile_pad   = 0,
        pre_pad    = 0,
        half       = False,
        device     = device,
    )
    upsampler.model.train()
    for p in upsampler.model.parameters():
        p.requires_grad = True
    return upsampler


device_obj = torch.device(f"cuda:0"
                           if torch.cuda.is_available() else "cpu")
print("\n✅ NWD Loss, SRQualityLoss, build_esrgan ready")

⬇️  Downloading ESRGAN 2x...
   ✅ 67 MB
⬇️  Downloading ESRGAN 4x...
   ✅ 67 MB

✅ NWD Loss, SRQualityLoss, build_esrgan ready


In [9]:
# ============================================================
# CELL 7 — FAST JointTileDataset (Offline SR Version)
#
# MODIFIED:
#   ✅ Reads pre-generated SR images directly
#   ✅ No live ESRGAN generation
#   ✅ Much faster training
#   ✅ Compatible with YOLOv8 / RT-DETR
# ============================================================

import cv2
import torch
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset


class JointTileDataset(Dataset):

    def __init__(self,
                 img_dir,
                 lbl_dir,
                 max_boxes=50):

        self.img_dir   = Path(img_dir)
        self.lbl_dir   = Path(lbl_dir)
        self.max_boxes = max_boxes
        self.hr_size   = 640

        # Only keep images with labels
        all_imgs = sorted(self.img_dir.glob("*.jpg"))

        self.imgs = []

        for p in all_imgs:
            lbl = self.lbl_dir / (p.stem + ".txt")

            if lbl.exists() and lbl.read_text().strip():
                self.imgs.append(p)

        print(
            f"Loaded {len(self.imgs)} tiles from:\n"
            f"   {self.img_dir}"
        )

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):

        img_path = self.imgs[idx]
        lbl_path = self.lbl_dir / (img_path.stem + ".txt")

        # =====================================================
        # IMAGE
        # =====================================================

        img = cv2.imread(str(img_path))

        if img is None:
            img = np.zeros((640,640,3), dtype=np.uint8)

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img = cv2.resize(
            img,
            (640,640),
            interpolation=cv2.INTER_LINEAR
        )

        # Tensor
        img_tensor = (
            torch.tensor(img, dtype=torch.float32)
            .permute(2,0,1)
            / 255.0
        )

        # =====================================================
        # LABELS
        # YOLO FORMAT:
        # cls cx cy w h
        # =====================================================

        box_tensor = torch.full(
            (self.max_boxes, 5),
            -1.0
        )

        count = 0

        if lbl_path.exists():

            lines = lbl_path.read_text().strip().split("\n")

            for line in lines:

                p = line.strip().split()

                if len(p) != 5:
                    continue

                if count >= self.max_boxes:
                    break

                try:

                    cls = int(p[0])

                    cx = float(p[1])
                    cy = float(p[2])
                    bw = float(p[3])
                    bh = float(p[4])

                    # sanity check
                    if not (
                        0 < bw < 1 and
                        0 < bh < 1
                    ):
                        continue

                    box_tensor[count] = torch.tensor([
                        cls,
                        cx,
                        cy,
                        bw,
                        bh
                    ])

                    count += 1

                except:
                    continue

        return {
            "image": img_tensor,
            "boxes": box_tensor
        }


# ============================================================
# DATASET PATHS
# ============================================================

TRAIN_IMG = "/kaggle/working/vd_tiles_sr/train/images"
TRAIN_LBL = "/kaggle/working/vd_tiles_sr/train/labels"

VAL_IMG   = "/kaggle/working/vd_tiles_sr/val/images"
VAL_LBL   = "/kaggle/working/vd_tiles_sr/val/labels"


# ============================================================
# DATASETS
# ============================================================

train_dataset = JointTileDataset(
    TRAIN_IMG,
    TRAIN_LBL
)

val_dataset = JointTileDataset(
    VAL_IMG,
    VAL_LBL
)

print("\nDatasets ready.")

Loaded 0 tiles from:
   /kaggle/working/vd_tiles_sr/train/images
Loaded 0 tiles from:
   /kaggle/working/vd_tiles_sr/val/images

Datasets ready.


In [10]:
# ============================================================
# CREATE YOLO DATASET YAML
# ============================================================

yaml_text = """
path: /kaggle/working/vd_tiles_sr

train: train/images
val: val/images

names:
  0: car
  1: van
  2: bus
  3: truck
  4: motor
  5: bicycle
"""

with open("/kaggle/working/vd_dataset.yaml", "w") as f:
    f.write(yaml_text)

print("✅ YAML created:")
print("/kaggle/working/vd_dataset.yaml")

✅ YAML created:
/kaggle/working/vd_dataset.yaml


In [11]:
# ── Add this disk space guard to the top of Cell 8 ───────────
import shutil as sh

def check_disk_gb():
    _, _, free = sh.disk_usage("/kaggle/working")
    return free / 1e9

# Add inside the loop in apply_esrgan_to_dataset():
# if check_disk_gb() < 2.0:
#     print(f"⚠️  Only {check_disk_gb():.1f}GB left — stopping!")
#     break

In [12]:
# ============================================================
# DOWNLOAD ESRGAN WEIGHTS
# ============================================================

import os
from pathlib import Path
from basicsr.utils.download_util import load_file_from_url

ROOT = Path("/root/.cache/realesrgan")

ROOT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# URLs
# ------------------------------------------------------------

URLS = {

    "x2": (
        "https://github.com/xinntao/Real-ESRGAN/"
        "releases/download/v0.2.1/"
        "RealESRGAN_x2plus.pth"
    ),

    "x4": (
        "https://github.com/xinntao/Real-ESRGAN/"
        "releases/download/v0.1.0/"
        "RealESRGAN_x4plus.pth"
    ),
}

# ------------------------------------------------------------
# Download
# ------------------------------------------------------------

for name, url in URLS.items():

    print(f"\n⬇️ Downloading {name} weights...")

    path = load_file_from_url(

        url=url,

        model_dir=str(ROOT),

        progress=True,
    )

    print(f"✅ Saved to: {path}")

print("\n🎉 ESRGAN weights ready")


⬇️ Downloading x2 weights...
Downloading: "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth" to /root/.cache/realesrgan/RealESRGAN_x2plus.pth



100%|██████████| 64.0M/64.0M [00:00<00:00, 158MB/s]


✅ Saved to: /root/.cache/realesrgan/RealESRGAN_x2plus.pth

⬇️ Downloading x4 weights...
Downloading: "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth" to /root/.cache/realesrgan/RealESRGAN_x4plus.pth



100%|██████████| 63.9M/63.9M [00:00<00:00, 169MB/s]

✅ Saved to: /root/.cache/realesrgan/RealESRGAN_x4plus.pth

🎉 ESRGAN weights ready


In [13]:
# ============================================================
# TEST ESRGAN — P100 safe
# ============================================================
import torch
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CPU    = torch.device("cpu")

# Detect if GPU supports pixel_unshuffle (Pascal/P100 does NOT)
def gpu_supports_esrgan():
    if not torch.cuda.is_available():
        return False
    cap = torch.cuda.get_device_capability()
    return cap[0] >= 7   # Turing (sm_70+) and above

ESRGAN_DEVICE = DEVICE if gpu_supports_esrgan() else CPU
print(f"GPU capability: {torch.cuda.get_device_capability() if torch.cuda.is_available() else 'N/A'}")
print(f"ESRGAN will run on: {ESRGAN_DEVICE}")
print(f"Detector will run on: {DEVICE}")

scale = 2
model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                num_block=23, num_grow_ch=32, scale=scale)
upsampler = RealESRGANer(
    scale=scale,
    model_path="/root/.cache/realesrgan/RealESRGAN_x2plus.pth",
    model=model,
    tile=0, tile_pad=10, pre_pad=0,
    half=False,                    # half=True crashes on P100
    device=ESRGAN_DEVICE           # CPU on P100, GPU on T4/A100
)
print("✅ ESRGAN LOADED SUCCESSFULLY")

GPU capability: (7, 5)
ESRGAN will run on: cuda
Detector will run on: cuda
✅ ESRGAN LOADED SUCCESSFULLY


In [14]:
# ============================================================
# OFFLINE ESRGAN GENERATION
# ============================================================

from pathlib import Path
import cv2
from tqdm.notebook import tqdm

INPUT_DIR  = Path("/kaggle/working/vd_tiles/train/images")
OUTPUT_DIR = Path("/kaggle/working/vd_tiles_sr/train/images")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Generating ESRGAN outputs offline...")

for img_path in tqdm(list(INPUT_DIR.glob("*.jpg"))):

    img = cv2.imread(str(img_path))

    # Replace with your ESRGAN inference
    sr = cv2.resize(
        img,
        (img.shape[1]*2, img.shape[0]*2),
        interpolation=cv2.INTER_CUBIC
    )

    cv2.imwrite(str(OUTPUT_DIR / img_path.name), sr)

print("Offline SR generation complete.")

Generating ESRGAN outputs offline...


  0%|          | 0/3459 [00:00<?, ?it/s]

Offline SR generation complete.


In [15]:
import shutil
from pathlib import Path

SRC = Path("/kaggle/working/vd_tiles/train/labels")
DST = Path("/kaggle/working/vd_tiles_sr/train/labels")

DST.mkdir(parents=True, exist_ok=True)

for f in SRC.glob("*.txt"):
    shutil.copy(f, DST/f.name)

print("Done")

Done


In [16]:
from pathlib import Path

base = Path("/kaggle/working/vd_tiles_sr")

print("\nTRAIN:")
print((base/"train").exists())
print((base/"train/images").exists())
print((base/"train/labels").exists())

print("\nVAL:")
print((base/"val").exists())
print((base/"val/images").exists())
print((base/"val/labels").exists())


TRAIN:
True
True
True

VAL:
False
False
False


In [17]:
# ============================================================
# GENERATE VAL SR IMAGES
# ============================================================

import cv2
import shutil
from pathlib import Path
from tqdm.notebook import tqdm

# INPUT
VAL_INPUT_IMG = Path("/kaggle/working/vd_tiles/val/images")
VAL_INPUT_LBL = Path("/kaggle/working/vd_tiles/val/labels")

# OUTPUT
VAL_SR_IMG = Path("/kaggle/working/vd_tiles_sr/val/images")
VAL_SR_LBL = Path("/kaggle/working/vd_tiles_sr/val/labels")

VAL_SR_IMG.mkdir(parents=True, exist_ok=True)
VAL_SR_LBL.mkdir(parents=True, exist_ok=True)

print("Generating validation SR images...")

for img_path in tqdm(list(VAL_INPUT_IMG.glob("*.jpg"))):

    img = cv2.imread(str(img_path))

    if img is None:
        continue

    # =====================================================
    # TEMPORARY SR
    # Replace with ESRGAN/SwinIR inference later
    # =====================================================

    sr = cv2.resize(
        img,
        (640,640),
        interpolation=cv2.INTER_CUBIC
    )

    cv2.imwrite(
        str(VAL_SR_IMG / img_path.name),
        sr
    )

print("Copying validation labels...")

for lbl in VAL_INPUT_LBL.glob("*.txt"):
    shutil.copy(lbl, VAL_SR_LBL / lbl.name)

print("\nValidation SR dataset ready.")

Generating validation SR images...


  0%|          | 0/259 [00:00<?, ?it/s]

Copying validation labels...

Validation SR dataset ready.


In [18]:
from pathlib import Path

print(len(list(Path("/kaggle/working/vd_tiles_sr/train/images").glob("*.jpg"))))
print(len(list(Path("/kaggle/working/vd_tiles_sr/val/images").glob("*.jpg"))))

3459
259


In [19]:
# ============================================================
# CELL A — FAST YOLOv8-L TRAINING (NO LIVE ESRGAN)
# ============================================================

import gc, torch, torch.nn as nn
from ultralytics import YOLO
from pathlib import Path

gc.collect()
torch.cuda.empty_cache()

DEVICE = torch.device("cuda")

print(f"Using GPU: {torch.cuda.get_device_name(0)}")

# ── Paths ────────────────────────────────────────────────────
VD_TILES = Path("/kaggle/working/vd_tiles_sr")

# ── Load YOLOv8-L ────────────────────────────────────────────
print("Loading YOLOv8-L...")
model = YOLO("yolov8l.pt")

# ── Train directly on SR images ──────────────────────────────
model.train(
    data="/kaggle/working/vd_dataset.yaml",
    epochs=15,
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    amp=True,
    cache=True,
    project="/kaggle/working/runs_fast",
    name="yolov8_sr_fast"
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Using GPU: Tesla T4
Loading YOLOv8-L...
Ultralytics 8.4.66 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/vd_dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ad001d40e00>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
     

In [20]:
# ============================================================
# CELL 11 — FAST NWD TRAINING
# OFFLINE SR VERSION
# ============================================================

import gc
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from pathlib import Path
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader

from ultralytics import YOLO

# ============================================================
# SETUP
# ============================================================

torch.backends.cudnn.benchmark = True

gc.collect()
torch.cuda.empty_cache()

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Using GPU: {torch.cuda.get_device_name(0)}")

# ============================================================
# PATHS
# ============================================================

VD_TILES = Path("/kaggle/working/vd_tiles_sr")
AT_TILES = Path("/kaggle/working/aitod_tiles_sr")

RUNS_DIR = Path("/kaggle/working/runs_fast")

RUNS_DIR.mkdir(exist_ok=True)

# ============================================================
# HYPERPARAMETERS
# ============================================================

IMG_SIZE    = 640
BATCH_SIZE  = 8
NUM_WORKERS = 2

LR_DETECT   = 1e-4

MAX_BOXES   = 50
NUM_QUERIES = 100

NWD_C       = 12.8

EPOCHS_VD_2X = 15
EPOCHS_VD_4X = 8

EPOCHS_AT_2X = 10
EPOCHS_AT_4X = 5

# ============================================================
# EXPERIMENTS
# ============================================================

EXPERIMENTS = [

    {
        "name":"VisDrone_2x",
        "tile_dir":VD_TILES,
        "epochs":EPOCHS_VD_2X,
        "num_classes":6
    },

    {
        "name":"VisDrone_4x",
        "tile_dir":VD_TILES,
        "epochs":EPOCHS_VD_4X,
        "num_classes":6
    },

    {
        "name":"AITOD_2x",
        "tile_dir":AT_TILES,
        "epochs":EPOCHS_AT_2X,
        "num_classes":8
    },

    {
        "name":"AITOD_4x",
        "tile_dir":AT_TILES,
        "epochs":EPOCHS_AT_4X,
        "num_classes":8
    }
]

# ============================================================
# DATASET
# ============================================================

class JointTileDataset(Dataset):

    def __init__(self,
                 img_dir,
                 lbl_dir,
                 max_boxes=50):

        self.img_dir   = Path(img_dir)
        self.lbl_dir   = Path(lbl_dir)

        self.max_boxes = max_boxes

        self.imgs = sorted(
            list(self.img_dir.glob("*.jpg"))
        )

        print(
            f"{self.img_dir}: "
            f"{len(self.imgs)} images"
        )

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):

        img_path = self.imgs[idx]

        lbl_path = self.lbl_dir / (
            img_path.stem + ".txt"
        )

        # IMAGE
        img = cv2.imread(str(img_path))

        if img is None:
            img = np.zeros(
                (640,640,3),
                dtype=np.uint8
            )

        img = cv2.cvtColor(
            img,
            cv2.COLOR_BGR2RGB
        )

        img = cv2.resize(
            img,
            (640,640)
        )

        img_tensor = (
            torch.tensor(
                img,
                dtype=torch.float32
            ).permute(2,0,1) / 255.0
        )

        # LABELS
        boxes = torch.full(
            (self.max_boxes,5),
            -1.0
        )

        if lbl_path.exists():

            lines = lbl_path.read_text().splitlines()

            count = 0

            for line in lines:

                if count >= self.max_boxes:
                    break

                p = line.strip().split()

                if len(p) != 5:
                    continue

                try:

                    cls = int(p[0])

                    cx = float(p[1])
                    cy = float(p[2])
                    bw = float(p[3])
                    bh = float(p[4])

                    boxes[count] = torch.tensor([
                        cls,
                        cx,
                        cy,
                        bw,
                        bh
                    ])

                    count += 1

                except:
                    continue

        return {
            "image": img_tensor,
            "boxes": boxes
        }

# ============================================================
# NWD LOSS
# ============================================================

def nwd_loss(pred_boxes, gt_boxes, C=12.8):

    if len(gt_boxes) == 0:
        return torch.tensor(
            0.0,
            device=pred_boxes.device
        )

    mu_p  = pred_boxes[:, :2]
    sig_p = pred_boxes[:, 2:] / 2

    mu_g  = gt_boxes[:, :2]
    sig_g = gt_boxes[:, 2:] / 2

    cd = (
        (
            mu_p.unsqueeze(1)
            -
            mu_g.unsqueeze(0)
        )**2
    ).sum(-1)

    sd = (
        (
            sig_p.unsqueeze(1)
            -
            sig_g.unsqueeze(0)
        )**2
    ).sum(-1)

    w2 = torch.sqrt(cd + sd + 1e-7)

    nwd = torch.exp(-w2 / C)

    best = nwd.max(dim=0).values

    return (1.0 - best).mean()

# ============================================================
# DETECTOR
# ============================================================

class YOLODetector(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        print("Loading YOLOv8-L backbone...")

        backbone = YOLO(
            "yolov8l.pt"
        ).model.model

        self.layers = nn.Sequential(
            *list(backbone.children())[:10]
        )

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512,512),
            nn.ReLU(True)
        )

        self.box_head = nn.Linear(
            512,
            NUM_QUERIES * 4
        )

        self.cls_head = nn.Linear(
            512,
            NUM_QUERIES * num_classes
        )

        self.num_queries = NUM_QUERIES
        self.num_classes = num_classes

    def forward(self, x):

        feat = x

        for layer in self.layers:
            feat = layer(feat)

        feat = self.pool(feat)

        feat = self.fc(feat)

        B = x.shape[0]

        boxes = torch.sigmoid(
            self.box_head(feat)
        ).view(B, NUM_QUERIES, 4)

        logits = self.cls_head(
            feat
        ).view(B, NUM_QUERIES, self.num_classes)

        return boxes, logits

# ============================================================
# TRAINER
# ============================================================

class Trainer:

    def __init__(self, num_classes):

        self.detector = YOLODetector(
            num_classes
        ).to(DEVICE)

        self.cls_loss = nn.CrossEntropyLoss()

        self.optimizer = torch.optim.AdamW(
            self.detector.parameters(),
            lr=LR_DETECT
        )

        self.history = {
            "L_total":[],
            "L_nwd":[],
            "L_cls":[]
        }

# ============================================================
# READY
# ============================================================

print("\n✅ FAST CELL 11 READY")
print("Offline SR training enabled.")

Using GPU: Tesla T4

✅ FAST CELL 11 READY
Offline SR training enabled.


In [ ]:
# ============================================================
# CELL — YOLOv12-L + ESRGAN Joint Training (NWD Loss)
# Experiments: VisDrone_2x, VisDrone_4x, AITOD_2x, AITOD_4x
# ============================================================

import os, gc, cv2, torch
import torch.nn as nn
import torch.nn.functional as F             
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# ── Ultralytics import ────────────────────────────────────────
from ultralytics import YOLO

torch.backends.cudnn.benchmark = True
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Paths (from Cell 2) ───────────────────────────────────────
VD_TILES = Path("/kaggle/working/vd_tiles")
AT_TILES = Path("/kaggle/working/aitod_tiles")
RUNS_DIR = Path("/kaggle/working/runs_yolov12l")
RUNS_DIR.mkdir(exist_ok=True)

# ── Hyperparameters ───────────────────────────────────────────
IMG_SIZE    = 640
BATCH_SIZE  = 1
GRAD_ACCUM  = 4
NUM_WORKERS = 2
LR_ESRGAN   = 1e-4
LR_DETECT   = 1e-4
LAMBDA_SR   = 0.1
NWD_C       = 12.8
MAX_BOXES   = 50

EPOCHS_VD_2X = 15
EPOCHS_VD_4X = 8
EPOCHS_AT_2X = 10
EPOCHS_AT_4X = 5

EXPERIMENTS_V12 = [
    {"name":"VisDrone_2x","tile_dir":VD_TILES,"sr_scale":2,"epochs":EPOCHS_VD_2X,"num_classes":6},
    {"name":"VisDrone_4x","tile_dir":VD_TILES,"sr_scale":4,"epochs":EPOCHS_VD_4X,"num_classes":6},
    {"name":"AITOD_2x",   "tile_dir":AT_TILES,"sr_scale":2,"epochs":EPOCHS_AT_2X,"num_classes":8},
    {"name":"AITOD_4x",   "tile_dir":AT_TILES,"sr_scale":4,"epochs":EPOCHS_AT_4X,"num_classes":8},
]

# JointTileDataset is defined in Cell 7 — remove the class definition below entirely
# (delete everything from "class JointTileDataset(Dataset):" down to "return {"lr": lr, ...}")

# =============================================================
#  JointTileDataset (unchanged from previous correct version)
# =============================================================
class JointTileDataset(Dataset):
    """
    Dataset for joint SR+detection training.
    Returns:
        lr   : low‑res image (3, H//scale, W//scale) in [0,1]
        hr   : original high‑res image (3, H, W) in [0,1]
        boxes: (MAX_BOXES, 5) – [class, cx, cy, w, h] (norm., pad with -1)
    """
    def __init__(self, img_dir, lbl_dir, sr_scale, max_boxes=50):
        self.img_dir   = Path(img_dir)
        self.lbl_dir   = Path(lbl_dir)
        self.sr_scale  = sr_scale
        self.max_boxes = max_boxes

        self.img_files = sorted(list(self.img_dir.glob("*.jpg")) +
                                list(self.img_dir.glob("*.png")))
        # filter images that have a corresponding label file
        self.samples = []
        for img_path in self.img_files:
            lbl_path = self.lbl_dir / (img_path.stem + ".txt")
            if lbl_path.exists():
                self.samples.append((img_path, lbl_path))
        if not self.samples:
            raise FileNotFoundError(
                f"No image‑label pairs found in {img_dir} / {lbl_dir}"
            )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, lbl_path = self.samples[idx]

        # Load HR image, keep RGB, normalise to [0,1]
        hr = Image.open(img_path).convert("RGB")
        w, h = hr.size

        # Downsample to LR (size = H//scale x W//scale)
        lr_size = (w // self.sr_scale, h // self.sr_scale)
        lr = hr.resize(lr_size, Image.BICUBIC)

        # Convert to tensors [0,1]
        hr = torch.from_numpy(np.array(hr).transpose(2,0,1)).float() / 255.0
        lr = torch.from_numpy(np.array(lr).transpose(2,0,1)).float() / 255.0

        # Read YOLO labels: class cx cy w h (normalised)
        boxes = np.full((self.max_boxes, 5), -1.0, dtype=np.float32)
        with open(lbl_path, "r") as f:
            lines = f.readlines()
        for i, line in enumerate(lines):
            if i >= self.max_boxes:
                break
            parts = line.strip().split()
            if len(parts) == 5:
                cls, cx, cy, bw, bh = map(float, parts)
                boxes[i] = [cls, cx, cy, bw, bh]

        boxes = torch.from_numpy(boxes)

        return {"lr": lr, "hr": hr, "boxes": boxes}


# ── NWD loss (unchanged) ──────────────────────────────────────
def nwd_loss(pred_boxes, gt_boxes, C=12.8):
    if pred_boxes.shape[0] == 0 or gt_boxes.shape[0] == 0:
        return torch.tensor(0.0, device=pred_boxes.device,
                            requires_grad=True)
    mu_p  = pred_boxes[:, :2];  sig_p = pred_boxes[:, 2:] / 2
    mu_g  = gt_boxes[:, :2];    sig_g = gt_boxes[:, 2:] / 2
    cd = ((mu_p.unsqueeze(1) - mu_g.unsqueeze(0)) ** 2).sum(-1)
    sd = ((sig_p.unsqueeze(1) - sig_g.unsqueeze(0)) ** 2).sum(-1)
    w2 = torch.sqrt(cd + sd + 1e-7)
    nwd_mat  = torch.exp(-w2 / C)
    best_nwd = nwd_mat.max(dim=0).values
    return (1.0 - best_nwd).mean()


# ── YOLOv12-L feature extractor wrapper (with download + channel detection) ─
class YOLOv12LBackbone(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()

        # --- Download YOLOv12-L weights if not present ---
        weight_file = "yolov12l.pt"
        if not os.path.isfile(weight_file):
            url = "https://github.com/sunsmarterjie/yolov12/releases/download/v1.0/yolov12l.pt"
            print(f"Downloading YOLOv12-L weights from {url} ...")
            torch.hub.download_url_to_file(url, weight_file)

        yolo = YOLO(weight_file)
        backbone = yolo.model.model

        # Keep layers 0-9 (backbone + neck C2f stages), drop the Detect head
        self.layers = nn.Sequential(*list(backbone.children())[:10])

        # Freeze all but the last 3 blocks
        params = list(self.layers.parameters())
        for p in params[:-6]:
            p.requires_grad = False

        # --- Determine output channels dynamically ---
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 640, 640)
            feat = dummy
            for layer in self.layers:
                feat = layer(feat)
            if isinstance(feat, (list, tuple)):
                feat = feat[-1]
            out_ch = feat.shape[1]          # e.g. 256 or 512
        print(f"  YOLOv12 backbone output channels: {out_ch}")

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.project = nn.Sequential(
            nn.Flatten(),
            nn.Linear(out_ch, 512),   # dynamic input dim
            nn.ReLU(inplace=True),
        )

        # Detection heads
        self.box_head = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(inplace=True),
            nn.Linear(256, 100 * 4)              # 100 queries × 4 coords
        )
        self.cls_head = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(inplace=True),
            nn.Linear(256, 100 * num_classes)
        )
        self.num_queries  = 100
        self.num_classes  = num_classes

    def forward(self, x):
        B = x.shape[0]
        feat = x
        for layer in self.layers:
            feat = layer(feat)
        if isinstance(feat, (list, tuple)):
            feat = feat[-1]
        feat = self.pool(feat)          # (B, C, 1, 1)
        feat = self.project(feat)       # (B, 512)
        boxes  = torch.sigmoid(
            self.box_head(feat).view(B, self.num_queries, 4))
        logits = self.cls_head(feat).view(
            B, self.num_queries, self.num_classes)
        return boxes, logits


# ── ESRGAN builder (unchanged) ───────────────────────────────
def build_esrgan_v12(scale, device):
    from realesrgan import RealESRGANer
    from basicsr.archs.rrdbnet_arch import RRDBNet
    from basicsr.utils.download_util import load_file_from_url
    url = (
        "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth"
        if scale == 2 else
        "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth"
    )
    path = load_file_from_url(url, "/root/.cache/realesrgan", progress=True)
    model = RRDBNet(num_in_ch=3, num_out_ch=3,
                    num_feat=64, num_block=23,
                    num_grow_ch=32, scale=scale)
    upsampler = RealESRGANer(
        scale=scale, model_path=path, model=model,
        tile=0, tile_pad=10, pre_pad=0,
        half=False, device=device)
    upsampler.model.train()
    for p in upsampler.model.parameters():
        p.requires_grad = True
    return upsampler


# ── Joint trainer with YOLOv12-L backbone (unchanged) ────────
class JointTrainerV12:
    def __init__(self, sr_scale, num_classes, device,
                 lr_esrgan, lr_detect, lambda_sr, nwd_c):
        self.device    = device
        self.best_loss = 1e9
        self.lambda_sr = lambda_sr
        self.nwd_c     = nwd_c
        self.num_cls   = num_classes
        self.history   = {"L_nwd": [], "L_sr": [],
                          "L_cls": [], "L_total": []}

        print(f"  [YOLOv12-L] Loading ESRGAN {sr_scale}×...")
        self.esrgan = build_esrgan_v12(
            sr_scale, device).model.to(device)

        print(f"  [YOLOv12-L] Loading YOLOv12-L backbone "
              f"({num_classes} classes)...")
        self.detector = YOLOv12LBackbone(num_classes).to(device)

        self.l1_loss  = nn.L1Loss()
        self.cls_loss = nn.CrossEntropyLoss(ignore_index=-1)

        self.opt_esrgan = torch.optim.AdamW(
            self.esrgan.parameters(),          lr=lr_esrgan)
        self.opt_det    = torch.optim.AdamW(
            filter(lambda p: p.requires_grad,
                   self.detector.parameters()), lr=lr_detect)

    def training_step(self, batch):
        lr    = batch["lr"].to(self.device)
        hr    = batch["hr"].to(self.device)
        boxes = batch["boxes"].to(self.device)

        sr = self.esrgan(lr)
        if sr.shape[-2:] != hr.shape[-2:]:
            sr = F.interpolate(sr, size=hr.shape[-2:],
                               mode="bilinear", align_corners=False)
        L_sr = self.l1_loss(sr, hr)

        pred_boxes, pred_logits = self.detector(sr)

        L_nwd = torch.tensor(0.0, device=self.device)
        L_cls = torch.tensor(0.0, device=self.device)
        B = lr.shape[0]
        valid_count = 0

        for b in range(B):
            gt       = boxes[b]
            mask     = gt[:, 0] >= 0
            if not mask.any():
                continue
            gt_valid  = gt[mask]
            gt_cls    = gt_valid[:, 0].long()
            gt_coords = gt_valid[:, 1:]
            gt_cls = torch.clamp(gt_cls, 0, self.num_cls - 1)

            pb = pred_boxes[b]
            pl = pred_logits[b]

            L_nwd = L_nwd + nwd_loss(pb, gt_coords, self.nwd_c)

            mu_p  = pb[:, :2];  sig_p = pb[:, 2:] / 2
            mu_g  = gt_coords[:, :2]; sig_g = gt_coords[:, 2:] / 2
            cd = ((mu_p.unsqueeze(1) - mu_g.unsqueeze(0)) ** 2).sum(-1)
            sd = ((sig_p.unsqueeze(1) - sig_g.unsqueeze(0)) ** 2).sum(-1)
            matched = torch.exp(
                -torch.sqrt(cd + sd + 1e-7) / self.nwd_c).argmax(dim=1)
            L_cls = L_cls + self.cls_loss(pl, gt_cls[matched])
            valid_count += 1

        if valid_count > 0:
            L_nwd = L_nwd / valid_count
            L_cls = L_cls / valid_count

        L_total = 2.0 * L_nwd + 1.0 * L_cls + self.lambda_sr * L_sr

        return {"L_total": L_total,
                "L_nwd":   L_nwd.detach(),
                "L_cls":   L_cls.detach(),
                "L_sr":    L_sr.detach()}

    def save(self, path, epoch):
        torch.save({"epoch":      epoch,
                    "esrgan":     self.esrgan.state_dict(),
                    "detector":   self.detector.state_dict(),
                    "best_loss":  self.best_loss,
                    "history":    self.history}, path)


# ============================================================
# MAIN TRAINING LOOP — YOLOv12-L + ESRGAN
# ============================================================

ALL_TRAINERS_V12 = {exp["name"]: None for exp in EXPERIMENTS_V12}

for exp in EXPERIMENTS_V12:
    exp_name   = exp["name"]
    tile_dir   = exp["tile_dir"]
    sr_scale   = exp["sr_scale"]
    n_epochs   = exp["epochs"]
    num_cls    = exp["num_classes"]

    print(f"\n{'='*62}")
    print(f"  [YOLOv12-L] Starting: {exp_name}")
    print(f"  SR scale={sr_scale}×  |  epochs={n_epochs}"
          f"  |  classes={num_cls}")
    print(f"{'='*62}")

    train_ds = JointTileDataset(
        img_dir   = tile_dir / "train" / "images",
        lbl_dir   = tile_dir / "train" / "labels",
        sr_scale  = sr_scale,
        max_boxes = MAX_BOXES,
    )
    val_ds = JointTileDataset(
        img_dir   = tile_dir / "val" / "images",
        lbl_dir   = tile_dir / "val" / "labels",
        sr_scale  = sr_scale,
        max_boxes = MAX_BOXES,
    )

    if len(train_ds) == 0:
        print(f"  ⚠️  No training tiles found for {exp_name} — skip")
        continue

    train_dl = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True,
        drop_last=True)
    val_dl   = DataLoader(
        val_ds,   batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True)

    trainer = JointTrainerV12(
        sr_scale  = sr_scale,
        num_classes = num_cls,
        device    = DEVICE,
        lr_esrgan = LR_ESRGAN,
        lr_detect = LR_DETECT,
        lambda_sr = LAMBDA_SR,
        nwd_c     = NWD_C,
    )
    ALL_TRAINERS_V12[exp_name] = trainer

    save_path = RUNS_DIR / f"{exp_name}_yolov12l_best.pt"

    for epoch in range(1, n_epochs + 1):
        trainer.esrgan.train()
        trainer.detector.train()
        epoch_losses = {"L_total": [], "L_nwd": [],
                        "L_cls": [],  "L_sr":  []}

        trainer.opt_esrgan.zero_grad()
        trainer.opt_det.zero_grad()

        pbar = tqdm(train_dl,
                    desc=f"  Epoch {epoch}/{n_epochs}",
                    leave=False)

        for step, batch in enumerate(pbar):
            losses = trainer.training_step(batch)

            (losses["L_total"] / GRAD_ACCUM).backward()

            for k, v in losses.items():
                epoch_losses[k].append(v.item())

            if (step + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(
                    list(trainer.esrgan.parameters()) +
                    list(filter(lambda p: p.requires_grad,
                                trainer.detector.parameters())),
                    max_norm=10.0)
                trainer.opt_esrgan.step()
                trainer.opt_det.step()
                trainer.opt_esrgan.zero_grad()
                trainer.opt_det.zero_grad()

            pbar.set_postfix(
                L=f"{losses['L_total'].item():.4f}",
                nwd=f"{losses['L_nwd'].item():.4f}",
                sr=f"{losses['L_sr'].item():.4f}")

        # ── Epoch summary ─────────────────────────────────────
        mean = {k: np.mean(v) for k, v in epoch_losses.items()}
        for k, v in mean.items():
            trainer.history[k].append(v)

        if mean["L_total"] < trainer.best_loss:
            trainer.best_loss = mean["L_total"]
            trainer.save(save_path, epoch)
            tag = " ✅ saved"
        else:
            tag = ""

        print(
            f"  [{exp_name}] Epoch {epoch:>2}/{n_epochs}  "
            f"L_total={mean['L_total']:.4f}  "
            f"L_nwd={mean['L_nwd']:.4f}  "
            f"L_cls={mean['L_cls']:.4f}  "
            f"L_sr={mean['L_sr']:.4f}"
            f"{tag}"
        )

        gc.collect()
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

    print(f"\n  ✅ {exp_name} complete — "
          f"best loss: {trainer.best_loss:.4f}")
    print(f"     Checkpoint: {save_path}")

print("\n" + "="*62)
print("  [YOLOv12-L] ALL EXPERIMENTS COMPLETE")
print("="*62)
for name, tr in ALL_TRAINERS_V12.items():
    if tr is None:
        print(f"  {name:<18}: skipped")
    else:
        hist = tr.history["L_total"]
        print(f"  {name:<18}: {len(hist)} epochs  "
              f"init={hist[0]:.4f}  final={hist[-1]:.4f}  "
              f"best={tr.best_loss:.4f}")


  [YOLOv12-L] Starting: VisDrone_2x
  SR scale=2×  |  epochs=15  |  classes=6
  [YOLOv12-L] Loading ESRGAN 2×...
  [YOLOv12-L] Loading YOLOv12-L backbone (6 classes)...


100%|██████████| 51.2M/51.2M [00:00<00:00, 216MB/s] 


  YOLOv12 backbone output channels: 512


  Epoch 1/15:   0%|          | 0/3459 [00:00<?, ?it/s]

  [VisDrone_2x] Epoch  1/15  L_total=1.3324  L_nwd=0.0095  L_cls=1.3111  L_sr=0.0240 ✅ saved


...

  Epoch 10/15:   0%|          | 0/3459 [00:00<?, ?it/s]

  [VisDrone_2x] Epoch 10/15  L_total=1.2976  L_nwd=0.0043  L_cls=1.2870  L_sr=0.0195


  Epoch 11/15:   0%|          | 0/3459 [00:00<?, ?it/s]

  [VisDrone_2x] Epoch 11/15  L_total=1.3001  L_nwd=0.0043  L_cls=1.2880  L_sr=0.0342


  Epoch 12/15:   0%|          | 0/3459 [00:00<?, ?it/s]

In [ ]:
# ============================================================
# CELL B — YOLOv12-L Visualization: IoU vs NWD
# Reads ALL_TRAINERS_V12_IOU + ALL_TRAINERS_V12
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

# Verify both dicts exist
assert "ALL_TRAINERS_V12_IOU" in dir(), \
   
assert "ALL_TRAINERS_V12" in dir(), \
    "Run Cell 12 first (YOLOv12 NWD training)"

EXP_NAMES  = ["VisDrone_2x","VisDrone_4x","AITOD_2x","AITOD_4x"]
SHORT      = ["VD 2×","VD 4×","AT 2×","AT 4×"]
IOU_COLOR  = "#E8734C"
NWD_COLOR  = "#4C9BE8"

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.patch.set_facecolor("#0F1117")
axes = axes.flatten()

# ── Plot 1-4: loss curves per experiment ─────────────────────
for pi, (exp_name, sname) in enumerate(zip(EXP_NAMES, SHORT)):
    ax = axes[pi]
    ax.set_facecolor("#1A1D27")
    ax.set_title(f"YOLOv12-L — {sname}",
                 color="white", fontweight="bold")
    ax.tick_params(colors="white")
    ax.spines[:].set_color("#333644")
     
    ax.set_axisbelow(True)

    # IoU curve (uses L_ciou key)
    tr_iou = ALL_TRAINERS_V12_IOU.get(exp_name)
    if tr_iou and tr_iou.history.get("L_ciou"):
        vals = tr_iou.history["L_ciou"]
        eps  = range(1, len(vals)+1)
        ax.plot(eps, vals, color=IOU_COLOR, linewidth=2.5,
                marker="o", markersize=4, label="CIoU Loss")
        ax.text(len(vals), vals[-1],
                f" {vals[-1]:.4f}", va="center",
                color=IOU_COLOR, fontsize=8)

    # NWD curve
    tr_nwd = ALL_TRAINERS_V12.get(exp_name)
    if tr_nwd and tr_nwd.history.get("L_nwd"):
        vals = tr_nwd.history["L_nwd"]
        eps  = range(1, len(vals)+1)
        ax.plot(eps, vals, color=NWD_COLOR, linewidth=2.5,
                marker="s", markersize=4, label="NWD Loss")
        ax.text(len(vals), vals[-1],
                f" {vals[-1]:.4f}", va="center",
                color=NWD_COLOR, fontsize=8)

    ax.set_xlabel("Epoch", color="white", fontsize=9)
    ax.set_ylabel("Box Loss", color="white", fontsize=9)
    ax.legend(facecolor="#1A1D27", labelcolor="white",
              edgecolor="#333644", fontsize=8)


# ── Plot 5: Final box loss bar comparison ────────────────────
ax5 = axes[4]
ax5.set_facecolor("#1A1D27")
ax5.tick_params(colors="white")
ax5.spines[:].set_color("#333644")
ax5.yaxis.grid(True, color="#2A2D3A",
               linestyle="--"); ax5.set_axisbelow(True)

x = np.arange(len(EXP_NAMES)); w = 0.35
iou_vals, nwd_vals = [], []

for exp_name in EXP_NAMES:
    tr_i = ALL_TRAINERS_V12_IOU.get(exp_name)
    tr_n = ALL_TRAINERS_V12.get(exp_name)
    iou_vals.append(
        tr_i.history["L_ciou"][-1]
        if tr_i and tr_i.history.get("L_ciou") else 0)
    nwd_vals.append(
        tr_n.history["L_nwd"][-1]
        if tr_n and tr_n.history.get("L_nwd") else 0)

b1 = ax5.bar(x-w/2, iou_vals, w, label="CIoU",
             color=IOU_COLOR, edgecolor="#0F1117")
b2 = ax5.bar(x+w/2, nwd_vals, w, label="NWD",
             color=NWD_COLOR, edgecolor="#0F1117")
for bar, v in zip(list(b1)+list(b2), iou_vals+nwd_vals):
    if v > 0:
        ax5.text(bar.get_x()+bar.get_width()/2,
                 v+0.001, f"{v:.4f}",
                 ha="center", color="white",
                 fontsize=7, fontweight="bold")
ax5.set_xticks(x); ax5.set_xticklabels(SHORT, color="white")
ax5.set_ylabel("Final Box Loss", color="white")
ax5.set_title("YOLOv12-L: CIoU vs NWD Final Loss",
              color="white", fontweight="bold")
ax5.legend(facecolor="#1A1D27", labelcolor="white",
           edgecolor="#333644")


# ── Plot 6: NWD improvement % ────────────────────────────────
ax6 = axes[5]
ax6.set_facecolor("#1A1D27")
ax6.tick_params(colors="white"); ax6.spines[:].set_color("#333644")
ax6.yaxis.grid(True, color="#2A2D3A", linestyle="--")
ax6.set_axisbelow(True)

improvements = [
    (iv-nv)/max(iv,1e-7)*100
    for iv,nv in zip(iou_vals, nwd_vals)]
bar_colors = ["#4CE87A" if i>0 else "#E84C4C"
               for i in improvements]
bars = ax6.bar(SHORT, improvements,
               color=bar_colors, edgecolor="#0F1117")
for bar, imp in zip(bars, improvements):
    ax6.text(bar.get_x()+bar.get_width()/2,
             imp + (0.3 if imp>=0 else -0.8),
             f"{imp:+.1f}%", ha="center",
             color="white", fontsize=9, fontweight="bold")
ax6.axhline(0, color="#999", linewidth=0.8)
ax6.set_ylabel("NWD improvement over CIoU (%)",
               color="white")
ax6.set_title("YOLOv12-L: NWD Gain Over CIoU",
              color="white", fontweight="bold")
ax6.tick_params(axis="x", colors="white")

plt.suptitle("YOLOv12-L + ESRGAN — CIoU vs NWD Loss Comparison\n"
             "VisDrone Tiny + AI-TOD | 2× and 4× SR",
             color="white", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("/kaggle/working/v12_iou_vs_nwd.png",
            dpi=150, bbox_inches="tight", facecolor="#0F1117")
plt.show()
print("✅ Saved → v12_iou_vs_nwd.png")

In [ ]:
# ============================================================
# CELL C — YOLO11-L + ESRGAN Joint Training (CIoU Loss)
# GPU-only. Depends on Cell 6 (build_esrgan) + Cell 7 (JointTileDataset)
# ============================================================

import gc, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader
from ultralytics import YOLO

assert torch.cuda.is_available(), "❌ GPU required"
torch.cuda.empty_cache(); gc.collect()
DEVICE = torch.device("cuda")

VD_TILES     = Path("/kaggle/working/vd_tiles")
AT_TILES     = Path("/kaggle/working/aitod_tiles")
RUNS_V11_IOU = Path("/kaggle/working/runs_yolo11l_iou")
RUNS_V11_IOU.mkdir(exist_ok=True)

BATCH_SIZE=1; GRAD_ACCUM=4; NUM_WORKERS=2; NUM_QUERIES=100
LR_ESRGAN=1e-4; LR_DETECT=1e-4; LAMBDA_SR=0.1; MAX_BOXES=50

EXPERIMENTS_V11_IOU = [
    {"name":"VisDrone_2x","tile_dir":VD_TILES,"sr_scale":2,"epochs":15,"num_classes":6},
    {"name":"VisDrone_4x","tile_dir":VD_TILES,"sr_scale":4,"epochs":8, "num_classes":6},
    {"name":"AITOD_2x",   "tile_dir":AT_TILES,"sr_scale":2,"epochs":10,"num_classes":8},
    {"name":"AITOD_4x",   "tile_dir":AT_TILES,"sr_scale":4,"epochs":5, "num_classes":8},
]

# ciou_loss defined in Cell A (Cell 10) — reused here automatically


class YOLO11LBackbone_IoU(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        print("  Loading yolo11l.pt ...")
        backbone = YOLO("yolo11l.pt").model.model           # auto-download via ultralytics
        self.layers = nn.Sequential(*list(backbone.children())[:10])
        for p in list(self.layers.parameters())[:-6]:
            p.requires_grad = False
        with torch.no_grad():
            f = torch.zeros(1,3,640,640)
            for layer in self.layers:
                f = layer(f)
                if isinstance(f,(list,tuple)): f=f[-1]
            if isinstance(f,(list,tuple)): f=f[-1]
            out_ch = f.shape[1]
        print(f"  YOLO11-IoU channels: {out_ch}")
        self.pool    = nn.AdaptiveAvgPool2d(1)
        self.project = nn.Sequential(nn.Flatten(),nn.Linear(out_ch,512),nn.ReLU(True))
        self.box_head= nn.Sequential(nn.Linear(512,256),nn.ReLU(True),nn.Linear(256,NUM_QUERIES*4))
        self.cls_head= nn.Sequential(nn.Linear(512,256),nn.ReLU(True),nn.Linear(256,NUM_QUERIES*num_classes))
        self.num_queries=NUM_QUERIES; self.num_classes=num_classes

    def forward(self, x):
        B=x.shape[0]; f=x
        for layer in self.layers:
            f=layer(f)
            if isinstance(f,(list,tuple)): f=f[-1]
        if isinstance(f,(list,tuple)): f=f[-1]
        emb    = self.project(self.pool(f))
        boxes  = torch.sigmoid(self.box_head(emb).view(B,self.num_queries,4))
        logits = self.cls_head(emb).view(B,self.num_queries,self.num_classes)
        return boxes, logits


class JointTrainerV11_IoU:
    def __init__(self, sr_scale, num_classes, lr_esrgan, lr_detect, lambda_sr):
        self.device=DEVICE; self.best_loss=1e9; self.lambda_sr=lambda_sr
        self.num_cls=num_classes
        self.history={"L_ciou":[],"L_sr":[],"L_cls":[],"L_total":[]}

        print(f"  ESRGAN {sr_scale}× on {DEVICE} ...")
        self.esrgan=build_esrgan(sr_scale,DEVICE).model.to(DEVICE)  # uses Cell 6 build_esrgan
        self.esrgan.train()
        for p in self.esrgan.parameters(): p.requires_grad=True

        print(f"  YOLO11-L backbone ({num_classes} cls) ...")
        self.det_head=YOLO11LBackbone_IoU(num_classes).to(DEVICE)

        self.l1=nn.L1Loss(); self.ce=nn.CrossEntropyLoss(ignore_index=-1)
        self.opt_esrgan=torch.optim.AdamW(self.esrgan.parameters(),lr=lr_esrgan)
        self.opt_det   =torch.optim.AdamW(
            filter(lambda p:p.requires_grad,self.det_head.parameters()),lr=lr_detect)

    def training_step(self, batch):
        lr=batch["lr"].to(DEVICE); hr=batch["hr"].to(DEVICE); boxes=batch["boxes"].to(DEVICE)
        sr=self.esrgan(lr)
        if sr.shape[-2:]!=hr.shape[-2:]:
            sr=F.interpolate(sr,size=hr.shape[-2:],mode="bilinear",align_corners=False)
        L_sr=self.l1(sr,hr)
        pred_boxes,pred_logits=self.det_head(sr)

        L_ciou=torch.zeros(1,device=DEVICE); L_cls=torch.zeros(1,device=DEVICE)
        B=lr.shape[0]; vc=0
        for b in range(B):
            gt=boxes[b]; mask=gt[:,0]>=0
            if not mask.any(): continue
            gt_v=gt[mask]
            gt_cls=torch.clamp(gt_v[:,0].long(),0,self.num_cls-1)
            gt_coords=gt_v[:,1:]
            pb=pred_boxes[b]; pl=pred_logits[b]
            Q=pb.shape[0]; M=gt_coords.shape[0]

            L_ciou=L_ciou+ciou_loss(pb,gt_coords)

            # IoU-argmax matching — NOT modulo
            def to_xyxy_loc(b_):
                return torch.stack([b_[:,0]-b_[:,2]/2,b_[:,1]-b_[:,3]/2,
                                    b_[:,0]+b_[:,2]/2,b_[:,1]+b_[:,3]/2],dim=-1)
            pb_xy=to_xyxy_loc(pb); gb_xy=to_xyxy_loc(gt_coords)
            pb_e=pb_xy.unsqueeze(1).expand(Q,M,4); gb_e=gb_xy.unsqueeze(0).expand(Q,M,4)
            inter=((torch.min(pb_e[...,2],gb_e[...,2])-torch.max(pb_e[...,0],gb_e[...,0])).clamp(0)*
                   (torch.min(pb_e[...,3],gb_e[...,3])-torch.max(pb_e[...,1],gb_e[...,1])).clamp(0))
            union=(pb[:,2]*pb[:,3]).unsqueeze(1)+(gt_coords[:,2]*gt_coords[:,3]).unsqueeze(0)-inter+1e-7
            matched=(inter/union).argmax(dim=1)
            L_cls=L_cls+self.ce(pl,gt_cls[matched]); vc+=1

        if vc>0: L_ciou=L_ciou/vc; L_cls=L_cls/vc
        L_total=2.0*L_ciou+1.0*L_cls+self.lambda_sr*L_sr
        return {"L_total":L_total,"L_ciou":L_ciou.detach(),
                "L_cls":L_cls.detach(),"L_sr":L_sr.detach()}

    def save(self, path, epoch):
        torch.save({"epoch":epoch,"esrgan":self.esrgan.state_dict(),
                    "det_head":self.det_head.state_dict(),
                    "best_loss":self.best_loss,"history":self.history}, path)


ALL_TRAINERS_V11_IOU = {e["name"]:None for e in EXPERIMENTS_V11_IOU}

for exp in EXPERIMENTS_V11_IOU:
    print(f"\n{'='*62}")
    print(f"  [YOLO11-IoU] {exp['name']}  SR={exp['sr_scale']}×  cls={exp['num_classes']}")
    print(f"{'='*62}")
    torch.cuda.empty_cache(); gc.collect()

    try:
        train_ds=JointTileDataset(img_dir=exp["tile_dir"]/"train"/"images",
                                   lbl_dir=exp["tile_dir"]/"train"/"labels",
                                   sr_scale=exp["sr_scale"],max_boxes=MAX_BOXES)
    except FileNotFoundError as e:
        print(f"  ❌ {e}"); continue
    if len(train_ds)==0: print("  ❌ No tiles"); continue

    dl=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,
                  num_workers=NUM_WORKERS,pin_memory=True,drop_last=True)
    trainer=JointTrainerV11_IoU(sr_scale=exp["sr_scale"],num_classes=exp["num_classes"],
                                 lr_esrgan=LR_ESRGAN,lr_detect=LR_DETECT,lambda_sr=LAMBDA_SR)
    ALL_TRAINERS_V11_IOU[exp["name"]]=trainer
    ckpt_dir=RUNS_V11_IOU/exp["name"]; ckpt_dir.mkdir(exist_ok=True)

    for epoch in range(1,exp["epochs"]+1):
        trainer.esrgan.train(); trainer.det_head.train()
        ep_l={"L_total":[],"L_ciou":[],"L_cls":[],"L_sr":[]}
        trainer.opt_esrgan.zero_grad(); trainer.opt_det.zero_grad()
        pbar=tqdm(dl,desc=f"  Ep {epoch}/{exp['epochs']}",leave=False)
        for step,batch in enumerate(pbar):
            m=trainer.training_step(batch)
            (m["L_total"]/GRAD_ACCUM).backward()
            for k,v in m.items():
                ep_l[k].append(v.item() if isinstance(v,torch.Tensor) else float(v))
            if (step+1)%GRAD_ACCUM==0:
                torch.nn.utils.clip_grad_norm_(
                    list(trainer.esrgan.parameters())+
                    list(filter(lambda p:p.requires_grad,trainer.det_head.parameters())),
                    max_norm=10.0)
                trainer.opt_esrgan.step(); trainer.opt_det.step()
                trainer.opt_esrgan.zero_grad(); trainer.opt_det.zero_grad()
            pbar.set_postfix(L=f"{m['L_total'].item():.4f}",
                             ciou=f"{m['L_ciou'].item():.4f}")
        mean={k:np.mean(v) for k,v in ep_l.items()}
        for k,v in mean.items(): trainer.history[k].append(v)
        tag=""
        if mean["L_total"]<trainer.best_loss:
            trainer.best_loss=mean["L_total"]
            trainer.save(ckpt_dir/"best.pt",epoch); tag=" ✅"
        print(f"  Ep{epoch:>2}  tot={mean['L_total']:.4f}  ciou={mean['L_ciou']:.4f}  "
              f"cls={mean['L_cls']:.4f}  sr={mean['L_sr']:.4f}{tag}")
        torch.cuda.empty_cache(); gc.collect()

    print(f"\n  ✅ {exp['name']} done — best={trainer.best_loss:.4f}")
print("\n🎉 YOLO11-L IoU ALL COMPLETE")

In [ ]:
# ============================================================
# CELL — YOLO11-L (YOLO26-L) + ESRGAN Joint Training (NWD)
# Experiments: VisDrone_2x, VisDrone_4x, AITOD_2x, AITOD_4x
# ============================================================

import os, gc, cv2, torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader
from ultralytics import YOLO

torch.backends.cudnn.benchmark = True
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VD_TILES    = Path("/kaggle/working/vd_tiles")
AT_TILES    = Path("/kaggle/working/aitod_tiles")
RUNS_DIR    = Path("/kaggle/working/runs_yolo11l")
RUNS_DIR.mkdir(exist_ok=True)

IMG_SIZE    = 640
BATCH_SIZE  = 1
GRAD_ACCUM  = 4
NUM_WORKERS = 2
LR_ESRGAN   = 1e-4
LR_DETECT   = 1e-4
LAMBDA_SR   = 0.1
NWD_C       = 12.8
MAX_BOXES   = 50

EPOCHS_VD_2X = 15
EPOCHS_VD_4X = 8
EPOCHS_AT_2X = 10
EPOCHS_AT_4X = 5

EXPERIMENTS_V11 = [
    {"name":"VisDrone_2x","tile_dir":VD_TILES,"sr_scale":2,"epochs":EPOCHS_VD_2X,"num_classes":6},
    {"name":"VisDrone_4x","tile_dir":VD_TILES,"sr_scale":4,"epochs":EPOCHS_VD_4X,"num_classes":6},
    {"name":"AITOD_2x",   "tile_dir":AT_TILES,"sr_scale":2,"epochs":EPOCHS_AT_2X,"num_classes":8},
    {"name":"AITOD_4x",   "tile_dir":AT_TILES,"sr_scale":4,"epochs":EPOCHS_AT_4X,"num_classes":8},
]

# JointTileDataset is defined in Cell 7 — remove the class definition below entirely
# (delete everything from "class JointTileDataset(Dataset):" down to "return {"lr": lr, ...}")# ── NWD loss ──────────────────────────────────────────────────
def nwd_loss(pred_boxes, gt_boxes, C=12.8):
    if pred_boxes.shape[0] == 0 or gt_boxes.shape[0] == 0:
        return torch.tensor(0.0, device=pred_boxes.device,
                            requires_grad=True)
    mu_p  = pred_boxes[:, :2];  sig_p = pred_boxes[:, 2:] / 2
    mu_g  = gt_boxes[:, :2];    sig_g = gt_boxes[:, 2:] / 2
    cd = ((mu_p.unsqueeze(1) - mu_g.unsqueeze(0))**2).sum(-1)
    sd = ((sig_p.unsqueeze(1) - sig_g.unsqueeze(0))**2).sum(-1)
    w2       = torch.sqrt(cd + sd + 1e-7)
    nwd_mat  = torch.exp(-w2 / C)
    best_nwd = nwd_mat.max(dim=0).values
    return (1.0 - best_nwd).mean()


# ── YOLO11-L backbone ─────────────────────────────────────────
class YOLO11LBackbone(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()

        weight_file = "yolo11l.pt"
        if not os.path.isfile(weight_file):
            print("⬇️  Downloading YOLO11-L weights...")
            torch.hub.download_url_to_file(
                "https://github.com/ultralytics/assets/releases/"
                "download/v8.3.0/yolo11l.pt",
                weight_file)
        else:
            print(f"✅ {weight_file} already present")

        yolo     = YOLO(weight_file)
        backbone = yolo.model.model

        self.layers = nn.Sequential(
            *list(backbone.children())[:10])

        # Freeze all but last 3 blocks
        params = list(self.layers.parameters())
        for p in params[:-6]:
            p.requires_grad = False

        # Detect output channels dynamically
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 640, 640)
            feat  = dummy
            for layer in self.layers:
                feat = layer(feat)
            if isinstance(feat, (list, tuple)):
                feat = feat[-1]
            out_ch = feat.shape[1]
        print(f"  YOLO11 backbone output channels: {out_ch}")

        self.pool    = nn.AdaptiveAvgPool2d(1)
        self.project = nn.Sequential(
            nn.Flatten(),
            nn.Linear(out_ch, 512),
            nn.ReLU(inplace=True))
        self.box_head = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(inplace=True),
            nn.Linear(256, 100 * 4))
        self.cls_head = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(inplace=True),
            nn.Linear(256, 100 * num_classes))
        self.num_queries = 100
        self.num_classes = num_classes

    def forward(self, x):
        B    = x.shape[0]
        feat = x
        for layer in self.layers:
            feat = layer(feat)
        if isinstance(feat, (list, tuple)):
            feat = feat[-1]
        feat   = self.pool(feat)
        feat   = self.project(feat)
        boxes  = torch.sigmoid(
            self.box_head(feat).view(B, self.num_queries, 4))
        logits = self.cls_head(feat).view(
            B, self.num_queries, self.num_classes)
        return boxes, logits


# ── ESRGAN builder ────────────────────────────────────────────
def build_esrgan_v11(scale, device):
    from realesrgan import RealESRGANer
    from basicsr.archs.rrdbnet_arch import RRDBNet
    from basicsr.utils.download_util import load_file_from_url
    url = (
        "https://github.com/xinntao/Real-ESRGAN/releases/"
        "download/v0.2.1/RealESRGAN_x2plus.pth"
        if scale == 2 else
        "https://github.com/xinntao/Real-ESRGAN/releases/"
        "download/v0.1.0/RealESRGAN_x4plus.pth"
    )
    path = load_file_from_url(
        url, "/root/.cache/realesrgan", progress=True)
    model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                    num_block=23, num_grow_ch=32, scale=scale)
    up = RealESRGANer(scale=scale, model_path=path, model=model,
                      tile=0, tile_pad=10, pre_pad=0,
                      half=False, device=device)
    up.model.train()
    for p in up.model.parameters():
        p.requires_grad = True
    return up


# ── Joint trainer ─────────────────────────────────────────────
class JointTrainerV11:
    def __init__(self, sr_scale, num_classes, device,
                 lr_esrgan, lr_detect, lambda_sr, nwd_c):
        self.device    = device
        self.best_loss = 1e9
        self.lambda_sr = lambda_sr
        self.nwd_c     = nwd_c
        self.num_cls   = num_classes
        self.history   = {"L_nwd":[],"L_sr":[],
                          "L_cls":[],"L_total":[]}

        print(f"  [YOLO11-L] Loading ESRGAN {sr_scale}×...")
        self.esrgan = build_esrgan_v11(
            sr_scale, device).model.to(device)

        print(f"  [YOLO11-L] Loading backbone "
              f"({num_classes} classes)...")
        self.detector = YOLO11LBackbone(num_classes).to(device)

        self.l1_loss  = nn.L1Loss()
        self.cls_loss = nn.CrossEntropyLoss(ignore_index=-1)

        self.opt_esrgan = torch.optim.AdamW(
            self.esrgan.parameters(), lr=lr_esrgan)
        self.opt_det    = torch.optim.AdamW(
            filter(lambda p: p.requires_grad,
                   self.detector.parameters()),
            lr=lr_detect)

    def training_step(self, batch):
        lr    = batch["lr"].to(self.device)
        hr    = batch["hr"].to(self.device)
        boxes = batch["boxes"].to(self.device)

        sr = self.esrgan(lr)
        if sr.shape[-2:] != hr.shape[-2:]:
            sr = F.interpolate(sr, size=hr.shape[-2:],
                               mode="bilinear",
                               align_corners=False)
        L_sr = self.l1_loss(sr, hr)

        pred_boxes, pred_logits = self.detector(sr)

        L_nwd = torch.tensor(0.0, device=self.device)
        L_cls = torch.tensor(0.0, device=self.device)
        B = lr.shape[0]; valid_count = 0

        for b in range(B):
            gt   = boxes[b]; mask = gt[:, 0] >= 0
            if not mask.any(): continue
            gt_v      = gt[mask]
            gt_cls    = torch.clamp(gt_v[:, 0].long(),
                                    0, self.num_cls-1)
            gt_coords = gt_v[:, 1:]
            pb = pred_boxes[b]; pl = pred_logits[b]

            L_nwd = L_nwd + nwd_loss(pb, gt_coords, self.nwd_c)

            mu_p  = pb[:,:2];          sig_p = pb[:,2:]/2
            mu_g  = gt_coords[:,:2];   sig_g = gt_coords[:,2:]/2
            cd = ((mu_p.unsqueeze(1)-mu_g.unsqueeze(0))**2).sum(-1)
            sd = ((sig_p.unsqueeze(1)-sig_g.unsqueeze(0))**2).sum(-1)
            matched = torch.exp(
                -torch.sqrt(cd+sd+1e-7)/self.nwd_c).argmax(dim=1)
            L_cls = L_cls + self.cls_loss(pl, gt_cls[matched])
            valid_count += 1

        if valid_count > 0:
            L_nwd = L_nwd / valid_count
            L_cls = L_cls / valid_count

        L_total = 2.0*L_nwd + 1.0*L_cls + self.lambda_sr*L_sr
        return {"L_total": L_total,
                "L_nwd":   L_nwd.detach(),
                "L_cls":   L_cls.detach(),
                "L_sr":    L_sr.detach()}

    def save(self, path, epoch):
        torch.save({"epoch":     epoch,
                    "esrgan":    self.esrgan.state_dict(),
                    "detector":  self.detector.state_dict(),
                    "best_loss": self.best_loss,
                    "history":   self.history}, path)


# ── Training loop ─────────────────────────────────────────────
ALL_TRAINERS_V11 = {exp["name"]: None for exp in EXPERIMENTS_V11}

for exp in EXPERIMENTS_V11:
    print(f"\n{'='*62}")
    print(f"  [YOLO11-L] Starting: {exp['name']}")
    print(f"  SR scale={exp['sr_scale']}×  |  "
          f"epochs={exp['epochs']}  |  classes={exp['num_classes']}")
    print(f"{'='*62}")

    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    try:
        train_ds = JointTileDataset(
            img_dir   = exp["tile_dir"]/"train"/"images",
            lbl_dir   = exp["tile_dir"]/"train"/"labels",
            sr_scale  = exp["sr_scale"],
            max_boxes = MAX_BOXES)
        val_ds = JointTileDataset(
            img_dir   = exp["tile_dir"]/"val"/"images",
            lbl_dir   = exp["tile_dir"]/"val"/"labels",
            sr_scale  = exp["sr_scale"],
            max_boxes = MAX_BOXES)
    except FileNotFoundError as e:
        print(f"  ⚠️  {e} — skipping"); continue

    if len(train_ds) == 0:
        print("  ⚠️  No tiles — skipping"); continue

    train_dl = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True,
        drop_last=True)

    trainer = JointTrainerV11(
        sr_scale    = exp["sr_scale"],
        num_classes = exp["num_classes"],
        device      = DEVICE,
        lr_esrgan   = LR_ESRGAN,
        lr_detect   = LR_DETECT,
        lambda_sr   = LAMBDA_SR,
        nwd_c       = NWD_C)
    ALL_TRAINERS_V11[exp["name"]] = trainer

    save_path = RUNS_DIR / f"{exp['name']}_yolo11l_best.pt"

    for epoch in range(1, exp["epochs"] + 1):
        trainer.esrgan.train()
        trainer.detector.train()
        epoch_losses = {"L_total":[],"L_nwd":[],
                        "L_cls":[], "L_sr":[]}
        trainer.opt_esrgan.zero_grad()
        trainer.opt_det.zero_grad()

        pbar = tqdm(train_dl,
                    desc=f"  Epoch {epoch}/{exp['epochs']}",
                    leave=False)

        for step, batch in enumerate(pbar):
            losses = trainer.training_step(batch)
            (losses["L_total"] / GRAD_ACCUM).backward()

            for k, v in losses.items():
                epoch_losses[k].append(
                    v.item() if isinstance(v, torch.Tensor)
                    else float(v))

            if (step + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(
                    list(trainer.esrgan.parameters()) +
                    list(filter(lambda p: p.requires_grad,
                                trainer.detector.parameters())),
                    max_norm=10.0)
                trainer.opt_esrgan.step()
                trainer.opt_det.step()
                trainer.opt_esrgan.zero_grad()
                trainer.opt_det.zero_grad()

            pbar.set_postfix(
                L  =f"{losses['L_total'].item():.4f}",
                nwd=f"{losses['L_nwd'].item():.4f}",
                sr =f"{losses['L_sr'].item():.4f}")

        mean = {k: np.mean(v) for k, v in epoch_losses.items()}
        for k, v in mean.items():
            trainer.history[k].append(v)

        tag = ""
        if mean["L_total"] < trainer.best_loss:
            trainer.best_loss = mean["L_total"]
            trainer.save(save_path, epoch)
            tag = " ✅ saved"

        print(f"  [{exp['name']}] "
              f"Ep{epoch:>2}/{exp['epochs']}  "
              f"tot={mean['L_total']:.4f}  "
              f"nwd={mean['L_nwd']:.4f}  "
              f"cls={mean['L_cls']:.4f}  "
              f"sr={mean['L_sr']:.4f}{tag}")

        gc.collect()
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

    print(f"\n  ✅ {exp['name']} done — "
          f"best: {trainer.best_loss:.4f}")

print("\n" + "="*62)
print("  [YOLO11-L NWD] ALL EXPERIMENTS COMPLETE")
print("="*62)
for name, tr in ALL_TRAINERS_V11.items():
    if tr is None:
        print(f"  {name:<18}: skipped")
        continue
    hist = tr.history["L_total"]
    print(f"  {name:<18}: {len(hist)} epochs  "
          f"init={hist[0]:.4f}  "
          f"final={hist[-1]:.4f}  "
          f"best={tr.best_loss:.4f}")

In [ ]:
# ============================================================
# CELL D — YOLO11-L Visualization: IoU vs NWD
# Reads ALL_TRAINERS_V11_IOU + ALL_TRAINERS_V11
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

assert "ALL_TRAINERS_V11_IOU" in dir(), "Run Cell C first"
assert "ALL_TRAINERS_V11" in dir(),     "Run Cell 17 first"

EXP_NAMES = ["VisDrone_2x","VisDrone_4x","AITOD_2x","AITOD_4x"]
SHORT     = ["VD 2×","VD 4×","AT 2×","AT 4×"]
IOU_COLOR = "#E8734C"; NWD_COLOR = "#4C9BE8"

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.patch.set_facecolor("#0F1117")
axes = axes.flatten()

for pi, (exp_name, sname) in enumerate(zip(EXP_NAMES, SHORT)):
    ax = axes[pi]
    ax.set_facecolor("#1A1D27")
    ax.set_title(f"YOLO11-L — {sname}",
                 color="white", fontweight="bold")
    ax.tick_params(colors="white")
    ax.spines[:].set_color("#333644")
    ax.yaxis.grid(True,color="#2A2D3A",
                  linestyle="--",linewidth=0.7)
    ax.set_axisbelow(True)

    tr_iou = ALL_TRAINERS_V11_IOU.get(exp_name)
    if tr_iou and tr_iou.history.get("L_ciou"):
        vals = tr_iou.history["L_ciou"]
        ax.plot(range(1,len(vals)+1), vals,
                color=IOU_COLOR, linewidth=2.5,
                marker="o", markersize=4, label="CIoU")
        ax.text(len(vals),vals[-1],f" {vals[-1]:.4f}",
                va="center",color=IOU_COLOR,fontsize=8)

    tr_nwd = ALL_TRAINERS_V11.get(exp_name)
    if tr_nwd and tr_nwd.history.get("L_nwd"):
        vals = tr_nwd.history["L_nwd"]
        ax.plot(range(1,len(vals)+1), vals,
                color=NWD_COLOR, linewidth=2.5,
                marker="s", markersize=4, label="NWD")
        ax.text(len(vals),vals[-1],f" {vals[-1]:.4f}",
                va="center",color=NWD_COLOR,fontsize=8)

    ax.set_xlabel("Epoch",color="white",fontsize=9)
    ax.set_ylabel("Box Loss",color="white",fontsize=9)
    ax.legend(facecolor="#1A1D27",labelcolor="white",
              edgecolor="#333644",fontsize=8)

# Bar comparison
ax5 = axes[4]; ax5.set_facecolor("#1A1D27")
ax5.tick_params(colors="white"); ax5.spines[:].set_color("#333644")
ax5.yaxis.grid(True,color="#2A2D3A",linestyle="--")
ax5.set_axisbelow(True)
x=np.arange(len(EXP_NAMES)); w=0.35

iou_v=[ALL_TRAINERS_V11_IOU[n].history["L_ciou"][-1]
       if ALL_TRAINERS_V11_IOU.get(n) and
       ALL_TRAINERS_V11_IOU[n].history.get("L_ciou") else 0
       for n in EXP_NAMES]
nwd_v=[ALL_TRAINERS_V11[n].history["L_nwd"][-1]
       if ALL_TRAINERS_V11.get(n) and
       ALL_TRAINERS_V11[n].history.get("L_nwd") else 0
       for n in EXP_NAMES]

b1=ax5.bar(x-w/2,iou_v,w,label="CIoU",
           color=IOU_COLOR,edgecolor="#0F1117")
b2=ax5.bar(x+w/2,nwd_v,w,label="NWD",
           color=NWD_COLOR,edgecolor="#0F1117")
for bar,v in zip(list(b1)+list(b2),iou_v+nwd_v):
    if v>0:
        ax5.text(bar.get_x()+bar.get_width()/2,v+0.001,
                 f"{v:.4f}",ha="center",color="white",
                 fontsize=7,fontweight="bold")
ax5.set_xticks(x); ax5.set_xticklabels(SHORT,color="white")
ax5.set_title("YOLO11-L: CIoU vs NWD Final Loss",
              color="white",fontweight="bold")
ax5.legend(facecolor="#1A1D27",labelcolor="white",
           edgecolor="#333644")
ax5.set_ylabel("Final Box Loss",color="white")

# Improvement bars
ax6=axes[5]; ax6.set_facecolor("#1A1D27")
ax6.tick_params(colors="white"); ax6.spines[:].set_color("#333644")
ax6.yaxis.grid(True,color="#2A2D3A",linestyle="--")
ax6.set_axisbelow(True)
imps=[(iv-nv)/max(iv,1e-7)*100 for iv,nv in zip(iou_v,nwd_v)]
bc=["#4CE87A" if i>0 else "#E84C4C" for i in imps]
bars=ax6.bar(SHORT,imps,color=bc,edgecolor="#0F1117")
for bar,imp in zip(bars,imps):
    ax6.text(bar.get_x()+bar.get_width()/2,
             imp+(0.3 if imp>=0 else -0.8),
             f"{imp:+.1f}%",ha="center",color="white",
             fontsize=9,fontweight="bold")
ax6.axhline(0,color="#999",linewidth=0.8)
ax6.set_title("YOLO11-L: NWD Gain Over CIoU",
              color="white",fontweight="bold")
ax6.set_ylabel("NWD improvement (%)",color="white")
ax6.tick_params(axis="x",colors="white")

plt.suptitle("YOLO11-L + ESRGAN — CIoU vs NWD Loss Comparison\n"
             "VisDrone Tiny + AI-TOD | 2× and 4× SR",
             color="white",fontsize=13,fontweight="bold")
plt.tight_layout()
plt.savefig("/kaggle/working/v11_iou_vs_nwd.png",
            dpi=150,bbox_inches="tight",facecolor="#0F1117")
plt.show()
print("✅ Saved → v11_iou_vs_nwd.png")

In [ ]:
# ============================================================
# CELL E — FINAL COMPARISON: 4 model variants
#
# YOLOv12-L CIoU  (ALL_TRAINERS_V12_IOU)
# YOLOv12-L NWD   (ALL_TRAINERS_V12)
# YOLO11-L  CIoU  (ALL_TRAINERS_V11_IOU)
# YOLO11-L  NWD   (ALL_TRAINERS_V11)
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

EXP_NAMES = ["VisDrone_2x","VisDrone_4x","AITOD_2x","AITOD_4x"]
SHORT     = ["VD 2×","VD 4×","AT 2×","AT 4×"]

VARIANTS = {
    "YOLOv12 CIoU": (ALL_TRAINERS_V12_IOU, "L_ciou", "#E8734C"),
    "YOLOv12 NWD" : (ALL_TRAINERS_V12,     "L_nwd",  "#4C9BE8"),
    "YOLO11 CIoU" : (ALL_TRAINERS_V11_IOU, "L_ciou", "#F39C12"),
    "YOLO11 NWD"  : (ALL_TRAINERS_V11,     "L_nwd",  "#2ECC71"),
}

fig = plt.figure(figsize=(24, 16))
fig.patch.set_facecolor("#0F1117")
gs  = gridspec.GridSpec(3, 4, figure=fig,
                         hspace=0.45, wspace=0.35)


# ── Rows 0-1: loss curves per experiment (2×2) ───────────────
for pi, (exp_name, sname) in enumerate(zip(EXP_NAMES, SHORT)):
    row = pi // 2
    col = pi % 2
    ax  = fig.add_subplot(gs[row, col])
    ax.set_facecolor("#1A1D27")
    ax.set_title(sname, color="white",
                 fontweight="bold", fontsize=11)
    ax.tick_params(colors="white", labelsize=7)
    ax.spines[:].set_color("#333644")
    ax.yaxis.grid(True, color="#2A2D3A",
                  linestyle="--", linewidth=0.6)
    ax.set_axisbelow(True)

    for vname, (trainers, loss_key, color) in VARIANTS.items():
        tr = trainers.get(exp_name) if trainers else None
        if tr is None or not tr.history.get(loss_key):
            continue
        vals = tr.history[loss_key]
        eps  = range(1, len(vals)+1)
        ax.plot(eps, vals, color=color, linewidth=2,
                label=vname, marker="o", markersize=3)
        ax.text(len(vals), vals[-1],
                f" {vals[-1]:.4f}", va="center",
                color=color, fontsize=6)

    ax.set_xlabel("Epoch", color="white", fontsize=8)
    ax.set_ylabel("Box Loss", color="white", fontsize=8)
    if pi == 0:
        ax.legend(facecolor="#1A1D27", labelcolor="white",
                  edgecolor="#333644", fontsize=7,
                  loc="upper right")


# ── Rows 0-1 col 2-3: grouped bar chart ──────────────────────
ax_bar = fig.add_subplot(gs[0:2, 2:4])
ax_bar.set_facecolor("#1A1D27")
ax_bar.tick_params(colors="white")
ax_bar.spines[:].set_color("#333644")
ax_bar.yaxis.grid(True, color="#2A2D3A", linestyle="--")
ax_bar.set_axisbelow(True)

n_vars  = len(VARIANTS)
x       = np.arange(len(EXP_NAMES))
total_w = 0.8
bar_w   = total_w / n_vars

for vi, (vname, (trainers, loss_key, color)) in \
        enumerate(VARIANTS.items()):
    vals = []
    for exp_name in EXP_NAMES:
        tr = trainers.get(exp_name) if trainers else None
        vals.append(
            tr.history[loss_key][-1]
            if tr and tr.history.get(loss_key) else 0)
    offset = (vi - n_vars/2 + 0.5) * bar_w
    bars   = ax_bar.bar(x + offset, vals, bar_w,
                        label=vname, color=color,
                        edgecolor="#0F1117")
    for bar, v in zip(bars, vals):
        if v > 0:
            ax_bar.text(
                bar.get_x() + bar.get_width()/2,
                v + 0.001, f"{v:.3f}",
                ha="center", color="white",
                fontsize=6, fontweight="bold",
                rotation=90)

ax_bar.set_xticks(x)
ax_bar.set_xticklabels(SHORT, color="white", fontsize=10)
ax_bar.set_ylabel("Final Box Loss", color="white")
ax_bar.set_title(
    "All Models — Final Box Loss Comparison",
    color="white", fontweight="bold", fontsize=12)
ax_bar.legend(facecolor="#1A1D27", labelcolor="white",
              edgecolor="#333644", fontsize=9)


# ── Row 2: summary table ──────────────────────────────────────
ax_tbl = fig.add_subplot(gs[2, :])
ax_tbl.axis("off")

rows = []
for vname, (trainers, loss_key, color) in VARIANTS.items():
    for exp_name, sname in zip(EXP_NAMES, SHORT):
        tr = trainers.get(exp_name) if trainers else None
        if tr is None or not tr.history.get(loss_key):
            rows.append([vname, sname, "—","—","—","—"])
            continue
        hist = tr.history[loss_key]
        tot  = tr.history["L_total"]
        drop = (tot[0]-tot[-1])/tot[0]*100 if tot else 0
        rows.append([
            vname, sname,
            f"{len(hist)}",
            f"{hist[0]:.4f}",
            f"{hist[-1]:.4f}",
            f"{drop:.1f}%"])

tbl = ax_tbl.table(
    cellText=rows,
    colLabels=["Model","Experiment","Epochs",
               "Init Loss","Final Loss","Total Drop %"],
    loc="center", cellLoc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(8.5)
tbl.scale(1, 1.6)

MDL_COLORS_TBL = {
    "YOLOv12 CIoU": "#E8734C",
    "YOLOv12 NWD" : "#4C9BE8",
    "YOLO11 CIoU" : "#F39C12",
    "YOLO11 NWD"  : "#2ECC71",
}

for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor("#2E75B6")
        cell.set_text_props(color="white", fontweight="bold")
    else:
        mdl = rows[r-1][0]
        bg  = MDL_COLORS_TBL.get(mdl, "#1A1D27")
        # 20% opacity tint
        cell.set_facecolor(bg + "33")
        cell.set_text_props(color="white")
    cell.set_edgecolor("#333644")


plt.suptitle(
    "FINAL COMPARISON — YOLOv12-L vs YOLO11-L\n"
    "CIoU Loss vs NWD Loss  |  "
    "Joint ESRGAN Training  |  "
    "VisDrone Tiny + AI-TOD  |  2× & 4× SR",
    color="white", fontsize=13, fontweight="bold")

plt.savefig("/kaggle/working/final_all_models_comparison.png",
            dpi=150, bbox_inches="tight",
            facecolor="#0F1117")
plt.show()
print("✅ Saved → final_all_models_comparison.png")


# ── Text rankings ─────────────────────────────────────────────
print("\n" + "="*65)
print("  FINAL RANKINGS — Best Final Box Loss per Experiment")
print("="*65)

for exp_name, sname in zip(EXP_NAMES, SHORT):
    print(f"\n  {sname}:")
    results = []
    for vname, (trainers, loss_key, _) in VARIANTS.items():
        tr = trainers.get(exp_name) if trainers else None
        if tr and tr.history.get(loss_key):
            results.append((vname, tr.history[loss_key][-1]))
    results.sort(key=lambda x: x[1])
    for rank, (vname, loss) in enumerate(results, 1):
        trophy = "🥇" if rank==1 else "🥈" if rank==2 \
                 else "🥉" if rank==3 else "  "
        print(f"    {trophy} {rank}. {vname:<18} {loss:.4f}")

print("\n" + "="*65)
print("  BEST MODEL OVERALL")
print("="*65)
all_results = []
for vname, (trainers, loss_key, _) in VARIANTS.items():
    exp_losses = []
    for exp_name in EXP_NAMES:
        tr = trainers.get(exp_name) if trainers else None
        if tr and tr.history.get(loss_key):
            exp_losses.append(tr.history[loss_key][-1])
    if exp_losses:
        all_results.append((vname, np.mean(exp_losses)))

all_results.sort(key=lambda x: x[1])
for rank, (vname, avg) in enumerate(all_results, 1):
    trophy = "🥇" if rank==1 else "🥈" if rank==2 \
             else "🥉" if rank==3 else "  "
    print(f"  {trophy} {rank}. {vname:<18} avg={avg:.4f}")
print("="*65)